# PADC (IEEE TWC 2023): CIFAR10 → ImageNet-1k Parquet → Kodak24 (Colab G4)

This notebook reuses the author's original `ADJSCC_V` / `OracleNet` network implementations.

**Research sequence**
1. CIFAR10 DeepJSCC-V **400 epochs**; CIFAR10 OraNet **150 epochs** (resume/skip if present).
2. **ImageNet 2012 validation 50,000 images**: deterministic **45,000 / 5,000** split, **128×128 center crops**; DeepJSCC-V warm-start from CIFAR10 and fine-tune for **50 epochs**.
3. Train a **separate ImageNet OraNet** for **150 epochs** (online teacher sampling for paper-like setting).
4. Test on **Kodak24 only after training**, with AWGN or a separately trained fading model, per-image PSNR, OraNet prediction, and CR selection.

### Input datasets (shared `datasets` folder via a shortcut in MyDrive)
- Share source: **Shared with me / datasets** (CIFAR10, CIFAR100, Kodak24, ImageNet-1k)
- In Google Drive, place **one shortcut named `datasets` in the root of your MyDrive**.
- In Colab: `/content/drive/MyDrive/datasets/` then `cifar-10-batches-py/`, `cifar-100-python/`, `kodak24/`, `imagenet-1k/data/`.
- The shared datasets are **read-only inputs**. No archive extraction, missing Kodak downloads, ImageNet cache creation, or training outputs are written into the shared source folder.
- ImageNet crop cache is saved under **your own** `MyDrive/<RUN_FOLDER>/dataset_cache/imagenet128/`, while checkpoints and results remain in `MyDrive/<RUN_FOLDER>/paper/awgn/`.

**Important:** You do **not** need the ImageNet `train-*.parquet` or `test-*.parquet` shards for the original paper's ImageNet/Kodak phase. These are **Parquet tables, not .zip/.tar.gz archives**.

**Storage:** Prepare a reusable, lossless (not JPEG-recompressed) RGB `uint8` 128×128 center-crop NPY on Drive (~2.46 GB), and copy it to Colab SSD when available. This is an implementation I/O optimization; it does not replace the original ImageNet images with another dataset.

**Evaluation caveat:** Original paper results also depend on unreported details and stochastic training. Numerically matching the reported PSNRs is not guaranteed by matching the public architecture and stated settings.


## 1. Configuration / reuse existing CIFAR checkpoint

- If CIFAR was trained with the previous **fresh** notebook, keep `RUN_FOLDER='PADC_reproduction_CIFAR_fresh_20261022_v1'`. Completed 400+150 epoch checkpoints are auto-detected; the same experiments are **not retrained from epoch 1**.
- If your existing CIFAR weights were trained under `MyDrive/PADC_reproduction/paper/awgn/checkpoints`, change `RUN_FOLDER` to `'PADC_reproduction'`.
- If no CIFAR checkpoints exist under this folder, this full notebook will first train them from scratch.
- Keep the same `RUN_FOLDER` after a Colab disconnection to resume `*_last.pt` for each stage.
- For a separate independent run, change `RUN_FOLDER`; don't delete the prior checkpoints.

**Shared folder:** In Google Drive, add a shortcut to the shared `datasets` folder at **My Drive root** (not inside a different `datasets` folder). The notebook checks that all datasets are visible before training. If the shortcut is missing or inaccessible in Colab, it will stop early with a precise diagnostic instead of silently downloading or overwriting data.


In [ ]:
# -------- EDIT ME: FULL PADC, CIFAR + IMAGENET PARQUET + KODAK --------
PROFILE = 'paper'            # full settings: 400 epochs DeepJSCC-V, 150 epochs OraNet
CHANNEL = 'awgn'             # first reproduce AWGN; fading requires a SEPARATE run
USE_GOOGLE_DRIVE = True      # checkpoints survive Colab disconnections

# Important: Use a NEW folder for an independent from-scratch experiment.
# Keep the SAME folder after a disconnect so resume=True restores last.pt.
RUN_FOLDER = 'PADC_reproduction_CIFAR_fresh_20261022_v1'

RUN_JSCC_TRAIN = True
RUN_ORACLE_TRAIN = True
ORACLE_TRAINING = 'online'   # resample SNR, CR and PSNR targets every epoch
RUN_CIFAR_EVALUATION = True  # set True to repeat CIFAR100 (not needed for ImageNet stage)

# Enable ImageNet transfer and Kodak24 held-out test.
RUN_KODAK_EVALUATION = True
RUN_IMAGENET_FINETUNE = True
RUN_ADJSCC_BASELINES = False

SEED = 2026
# Shortcut at MyDrive root pointing to Shared with me / datasets.
# Source folders may be VIEWER-only; the notebook never writes into them.
DATA_ROOT = '/content/drive/MyDrive/datasets'
CIFAR10_ROOT = DATA_ROOT
CIFAR100_ROOT = DATA_ROOT
KODAK_PATH = DATA_ROOT + '/kodak24'
IMAGENET_DIR = DATA_ROOT + '/imagenet-1k/data'   # must contain all 14 validation-*.parquet files
# Keep the 2.46 GB ImageNet RGB128 cache OUTSIDE the shared source folder.
# Empty = MyDrive/<RUN_FOLDER>/dataset_cache/imagenet128 (resolved after Drive mount).
IMAGENET_CACHE_DIR = ''
IMAGENET_CACHE_LOCAL = True  # fast /content mmap, persistent copy in Drive
IMAGENET_FORCE_REBUILD = False  # only True if deliberately rebuilding the 2.46 GB crop cache
IMAGENET_TRAIN_BATCH = 32       # paper. Reduce to 16 or 8 only if G4 CUDA OOM.
IMAGENET_TEACHER_BATCH = 16     # teacher labels; reduce to 8 if OOM
IMAGENET_ORACLE_TRAINING = 'online'  # faithful fresh labels; 'cached' much faster but changes training
IMAGENET_ORACLE_EPOCHS = 150    # paper describes same OraNet process as CIFAR10
KODAK_MC_REPEATS = 100         # paper evaluation. Set 1-5 for a separate diagnostic run, not paper comparison

AUTO_UNPACK_CIFAR_ARCHIVES = True   # only when needed: unpack into local /content scratch, NEVER shared Drive
DOWNLOAD_CIFAR_IF_MISSING = False  # set True only if no local CIFAR archives/folders

# Optional .npy path overrides; leave blank when using original CIFAR archives.
CIFAR10_TRAIN_NPY = ''
CIFAR10_TEST_NPY = ''
CIFAR100_TEST_NPY = ''

PIN_AUTHOR_COMMIT = '2b5fae05f1194189a97e478be21be1e2ae6d4c18'  # SHA from prior verified Colab run

print('Profile:', PROFILE, '| Channel:', CHANNEL, '| Run folder:', RUN_FOLDER)
print('ImageNet parquet path:', IMAGENET_DIR)
assert PROFILE in ('smoke', 'research', 'paper')
assert CHANNEL in ('awgn', 'slow_rayleigh', 'repo_fading')
assert ORACLE_TRAINING in ('cached', 'online')
assert RUN_FOLDER and '/' not in RUN_FOLDER and '\\' not in RUN_FOLDER

assert IMAGENET_ORACLE_TRAINING in ('online', 'cached')


In [ ]:
import os,sys,subprocess,platform,json,time,math,random,shutil
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from PIL import Image
from IPython.display import display

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive',force_remount=False)
    except ImportError as e:
        raise RuntimeError('USE_GOOGLE_DRIVE=True requires the Colab environment; set it False locally.') from e
OUTPUT_PARENT = Path('/content/drive/MyDrive' if USE_GOOGLE_DRIVE else '/content')
OUTPUT_ROOT = OUTPUT_PARENT / RUN_FOLDER
# Cache is writable and separated from Shared with me / datasets.
if not IMAGENET_CACHE_DIR:
    IMAGENET_CACHE_DIR = str(OUTPUT_ROOT / 'dataset_cache' / 'imagenet128')
_shared_root = str(Path(DATA_ROOT).expanduser())
_cache_root = str(Path(IMAGENET_CACHE_DIR).expanduser())
if os.path.commonpath([_shared_root, _cache_root]) == _shared_root:
    raise ValueError('IMAGENET_CACHE_DIR must NOT be inside shared datasets; use MyDrive/<RUN_FOLDER>/dataset_cache')
print('Shared input datasets (read only):', DATA_ROOT)
print('ImageNet crop cache (your Drive):', IMAGENET_CACHE_DIR)
EXPERIMENT_DIR = OUTPUT_ROOT / PROFILE / CHANNEL
EXPERIMENT_DIR.mkdir(parents=True,exist_ok=True)
RESULTS_DIR=EXPERIMENT_DIR/'results';RESULTS_DIR.mkdir(exist_ok=True,parents=True)
FIG_DIR=RESULTS_DIR/'figures';FIG_DIR.mkdir(exist_ok=True,parents=True)
print('Outputs:',EXPERIMENT_DIR)
print('Python:',sys.version.split()[0], 'PyTorch:',torch.__version__)
# A fresh RUN_FOLDER has no checkpoint files: first launch begins at epoch 1.
# The SAME folder on subsequent launches enables safe automatic checkpoint resume.
existing_last = sorted((EXPERIMENT_DIR / 'checkpoints').glob('*_last.pt'))
if existing_last:
    print('CHECKPOINTS FOUND: the training cells will resume from this run:')
    for ckpt in existing_last: print('  ', ckpt.name)
else:
    print('NEW EXPERIMENT: no last.pt checkpoints; training starts at epoch 1.')
print('Colab disconnection does not auto-reconnect. Rerun all cells with this RUN_FOLDER to resume.')


### Preflight: verify Drive checkpoint location

The notebook automatically resumes CIFAR training from `RUN_FOLDER/paper/awgn/checkpoints`. The 14 ImageNet validation Parquet shards are validated in section 9. No HF login or internet download is needed for those files because they are already in Google Drive.


In [ ]:
# Shared-dataset shortcut preflight.  RUN THIS AFTER THE DRIVE MOUNT CELL.
# Does not download, extract, modify, or delete any source files.
from pathlib import Path

shared = Path(DATA_ROOT).expanduser()
mydrive = Path('/content/drive/MyDrive')
print('SHARED DATA SOURCE:', shared)
print('Exists:', shared.exists(), '| Directory:', shared.is_dir())
if not shared.is_dir():
    seen = sorted(p.name for p in mydrive.iterdir()) if mydrive.is_dir() else []
    raise FileNotFoundError(
        'The shared `datasets` shortcut is not visible to Colab at ' + str(shared) + '\n'
        'Add shortcut: Shared with me > datasets > Organize > Add shortcut > My Drive (root).\n'
        'Then remount the SAME Google account and rerun. MyDrive entries: ' + repr(seen[:30])
    )
print('Root entries:', sorted(p.name for p in shared.iterdir())[:30])

c10_dir = shared / 'cifar-10-batches-py'
c100_dir = shared / 'cifar-100-python'
kodak_dir = shared / 'kodak24'
image_dir = shared / 'imagenet-1k' / 'data'
image_names = [f'validation-{i:05d}-of-00014.parquet' for i in range(14)]
missing_image_shards = [fn for fn in image_names if not (image_dir / fn).is_file()]

c10_extracted = all((c10_dir / fn).is_file() for fn in
                    ['data_batch_1','data_batch_2','data_batch_3','data_batch_4','data_batch_5','test_batch'])
c100_extracted = all((c100_dir / fn).is_file() for fn in ['train','test','meta'])
c10_archive = (shared / 'cifar-10-python.tar.gz').is_file()
c100_archive = (shared / 'cifar-100-python.tar.gz').is_file()
kodak_missing = [i for i in range(1,25) if not (kodak_dir / f'kodim{i:02d}.png').is_file()]

print('CIFAR10  : extracted=',c10_extracted,' archive=',c10_archive)
print('CIFAR100 : extracted=',c100_extracted,' archive=',c100_archive)
print('Kodak24  : visible=',24-len(kodak_missing),'/24 | missing=',kodak_missing)
print('ImageNet : validation parquet=',14-len(missing_image_shards),'/14 | dir=',image_dir)
if not (c10_extracted or c10_archive or CIFAR10_TRAIN_NPY):
    raise FileNotFoundError('No valid CIFAR10 train/test directory or archive in shared datasets.')
if not (c100_extracted or c100_archive or CIFAR100_TEST_NPY):
    raise FileNotFoundError('No valid CIFAR100 directory or archive in shared datasets.')
if RUN_IMAGENET_FINETUNE and missing_image_shards:
    raise FileNotFoundError(
        'Missing ImageNet validation parquet shards under the shared shortcut. '
        'Missing: ' + repr(missing_image_shards) + '\n'
        'Expected: ' + str(image_dir) + '\n'
        'Check Colab Drive mount/shortcut/permissions, not the parquet loader.'
    )
if RUN_KODAK_EVALUATION and kodak_missing and kodak_missing != [18]:
    raise FileNotFoundError('Shared Kodak24 is incomplete: ' + str(kodak_missing))
if c10_extracted and c100_extracted and not kodak_missing and not missing_image_shards:
    print('PASS: all four datasets are visible from the shared folder shortcut.')
else:
    print('Preflight passed for available inputs. Missing archives/one Kodak image handled later where supported.')


## 2. Get the **original GitHub model** and record its revision

Original weights/binaries are **not** copied into this notebook. Network components are directly imported from the author's `models.py` (Encoder, Decoder, AF blocks, residual GDN blocks) and `OracleNet.py` (residual FC). The local changes below fix batch=1, device handling, PSNR and channel semantics **without changing the author's trained parameter names**.

In [ ]:
AUTHOR_REPO_URL = 'https://github.com/wyzhang-ustb/Predictive-and-Adaptive-Deep-Coding-for-Wireless-Image-Transmission-in-Semantic-Communication.git'
AUTHOR_REPO = Path('/content/PADC_author_original')
if not (AUTHOR_REPO/'models.py').exists():
    subprocess.run(['git','clone','--depth','1',AUTHOR_REPO_URL,str(AUTHOR_REPO)],check=True)
if PIN_AUTHOR_COMMIT:
    # A full fetch is only needed if the hash is not already present.
    subprocess.run(['git','-C',str(AUTHOR_REPO),'fetch','--depth','1','origin',PIN_AUTHOR_COMMIT],check=True)
    subprocess.run(['git','-C',str(AUTHOR_REPO),'checkout','--detach',PIN_AUTHOR_COMMIT],check=True)
AUTHOR_SHA=subprocess.check_output(['git','-C',str(AUTHOR_REPO),'rev-parse','HEAD'],text=True).strip()
assert (AUTHOR_REPO/'OracleNet.py').exists() and (AUTHOR_REPO/'GDN.py').exists()
if str(AUTHOR_REPO) not in sys.path:sys.path.insert(0,str(AUTHOR_REPO))
import models as upstream_models
from OracleNet import OracleNet as UpstreamOracleNet
print('Upstream commit:',AUTHOR_SHA)
print('Author model:',upstream_models.ADJSCC_V, 'Oracle:',UpstreamOracleNet)

## 3. Reproduction functions — source reviewable and editable

The next cells are notebook-embedded Python, so **no external helper `.py` download is needed**. They do not redefine the original Encoder/Decoder/OracleNet parameter structures. Power/complex symbol conventions and mask quantization are made explicit. We retain the author's **uint8 truncation** convention for PSNR labels, and separately support a continuous-PSNR diagnostic function.

### Module: `padc_core.py`

In [ ]:
"""PADC notebook: integration with original author's Encoder/Decoder/OracleNet.

Not a replacement for the author's model architecture. It patches the batch=1
attention issue, replaces device-hardcoded channel operations, and makes the
paper CR explicit. Copyright for original network modules remains with authors.
"""
import math
import random
import hashlib
import os
from pathlib import Path
from contextlib import contextmanager

import numpy as np
import torch
from torch import nn


def seed_everything(seed=2026):
    random.seed(int(seed))
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = False
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True


def device_info():
    if not torch.cuda.is_available():
        print('WARNING: CUDA GPU not found. Smoke tests can run on CPU; full training needs a Colab GPU.')
        return torch.device('cpu')
    idx = torch.cuda.current_device()
    print('GPU:', torch.cuda.get_device_name(idx))
    print('CUDA device memory (GiB):', round(torch.cuda.get_device_properties(idx).total_memory / 2**30, 2))
    return torch.device('cuda', idx)


def patch_author_attention():
    """Reuse original AFB learned layers, fix only shape handling at B=1."""
    from models import AF_block
    if getattr(AF_block.forward, '_padc_safe', False):
        return
    def forward(self, x, snr):
        bs = x.size(0)
        snr = snr.reshape(bs, 1).to(device=x.device, dtype=x.dtype)
        pooled = x.mean(dim=(2, 3))
        weights = self.sigmoid(self.fc2(self.relu(self.fc1(torch.cat([pooled, snr], dim=1)))))
        return x * weights.reshape(bs, -1, 1, 1)
    forward._padc_safe = True
    AF_block.forward = forward


def create_jscc(n_conv=256, kernel_size=5, height=32, width=32, channels=48, device='cpu'):
    patch_author_attention()
    from models import ADJSCC_V
    if height % 4 != 0 or width % 4 != 0:
        raise ValueError('Both image dimensions must be divisible by four.')
    if (height // 4 * width // 4) % 2:
        raise ValueError('Latent spatial area must be even for complex I/Q pairing.')
    model = ADJSCC_V([channels, height // 4, width // 4], kernel_size, n_conv)
    return model.to(device)


def create_oracle(device='cpu'):
    from OracleNet import OracleNet
    return OracleNet(48).to(device)


def channel_count(rate, batch_size=None, device=None):
    """R = #complex symbols / #source RGB real values. R_max = 0.5.

    The original repo takes an internal fraction cr = 2R; its mask_gen keeps
    round(48*cr) latent channels. Thus N_active = round(96R).
    """
    if torch.is_tensor(rate):
        rr = rate.reshape(-1, 1).float()
        if device is not None:
            rr = rr.to(device)
    else:
        rr = torch.tensor([[float(rate)]], device=device)
    if batch_size is not None and rr.size(0) == 1 and batch_size > 1:
        rr = rr.expand(batch_size, 1)
    if batch_size is not None and rr.size(0) != batch_size:
        raise ValueError('rate and batch size mismatch')
    if ((rr <= 0) | (rr > 0.5 + 1e-7)).any():
        raise ValueError('Physical CR must lie in (0, 0.5], not the repo internal cr.')
    return torch.round(96.0 * rr).clamp(1, 48).long()


def effective_cr(n_active):
    return n_active.float() / 96.0


def feature_vector(z0, snr, n_active):
    """Exactly the features in author's OraNet: CWM, CWSTD, SNR, kept channels."""
    if z0.ndim != 4 or z0.size(1) != 48:
        raise ValueError('Require unmasked Bx48xhxw encoder features.')
    bs = z0.size(0)
    mu = z0.float().mean(dim=(2, 3))
    sd = z0.float().std(dim=(2, 3), unbiased=True)
    return torch.cat([mu, sd, snr.reshape(bs, 1).float(), n_active.reshape(bs, 1).float()], dim=1)


def oracle_forward(oracle, features):
    """Original author's RFCB architecture and weights; robust B=1 input shape."""
    out = oracle.fc1(features.float())
    out = oracle.fc2(out)
    return oracle.relu(oracle.fc3(out)).reshape(-1)


def _real_mask(n_active, spatial, device, dtype):
    batch = n_active.size(0)
    ids = torch.arange(48, device=device).view(1, 48, 1)
    return (ids < n_active.reshape(batch, 1, 1)).to(dtype).expand(batch, 48, spatial).reshape(batch, 48 * spatial)


def _complex_channel(z, snr, real_mask, mode, power, eps=1e-8):
    """Physical complex I/Q channel with per-symbol (repo) or one-block fade.

    X has real components with average power P, hence complex symbol power 2P.
    Each independent real AWGN component has variance P / (10**(SNR/10)).
    """
    bs, n = z.shape
    if n % 2:
        raise ValueError('I/Q length must be even')
    zi = z[:, 0::2]
    zq = z[:, 1::2]
    zz = torch.complex(zi, zq)
    # Use independent real and imaginary noise and mask inactive I/Q pairs.
    sigma = math.sqrt(power) * torch.pow(10.0, -snr.reshape(bs, 1).float() / 20)
    nr = torch.randn_like(zi) * sigma
    ni = torch.randn_like(zq) * sigma
    mask_c = real_mask[:, 0::2]
    noise = torch.complex(nr, ni) * mask_c
    if mode == 'slow_rayleigh':
        # Block fading h~CN(0,1), constant per image. Receiver has perfect CSI.
        hr = torch.randn(bs, 1, device=z.device, dtype=z.dtype)
        hi = torch.randn(bs, 1, device=z.device, dtype=z.dtype)
        h = torch.complex(hr, hi) / math.sqrt(2)
    elif mode == 'repo_fading':
        # Original repo uses independent h per symbol, CN(0,2), despite 'slow' label.
        hr = torch.randn_like(zi)
        hi = torch.randn_like(zq)
        h = torch.complex(hr, hi)
    else:
        raise ValueError(mode)
    # Almost-surely nonzero; epsilon affects only exceedingly deep fades.
    invh = h.conj() / h.abs().square().clamp_min(eps)
    rec = (h * zz + noise) * invh
    rec = rec * mask_c
    return torch.stack([rec.real, rec.imag], dim=-1).reshape(bs, n)


def masked_channel(z0_flat, n_active, snr, mode='awgn', power=1.0):
    """Corrected, per-image PADC channel with zero-filled missing channels."""
    bs, n = z0_flat.shape
    if n % 48:
        raise ValueError('Feature length must contain 48 spatial channels')
    spatial = n // 48
    if spatial % 2:
        raise ValueError('Latent H*W must be even for I/Q grouping')
    n_active = n_active.reshape(bs, 1)
    snr = snr.reshape(bs, 1).to(z0_flat.device)
    mask = _real_mask(n_active, spatial, z0_flat.device, z0_flat.dtype)
    selected = z0_flat * mask
    energy = selected.float().square().sum(dim=1, keepdim=True).clamp_min(1e-8)
    # Number of actually sent REAL symbols = channels * latent spatial pixels.
    n_real = n_active.float() * spatial
    scaled = selected * (power * n_real / energy).sqrt()
    if mode == 'awgn':
        sigma = math.sqrt(power) * torch.pow(10.0, -snr.float() / 20)
        return (scaled + torch.randn_like(scaled) * sigma * mask) * mask
    if mode in ('slow_rayleigh', 'repo_fading'):
        return _complex_channel(scaled, snr, mask, mode, power)
    if mode == 'noiseless':
        return scaled
    raise ValueError(f'Invalid channel {mode!r}')


def encode_features(jscc, x, snr):
    return jscc.encoder(x, snr.reshape(-1, 1)).reshape(x.size(0), 48, x.size(-2)//4, x.size(-1)//4)


def reconstruct_from_latent(jscc, latent, snr, n_active, mode='awgn', power=1.):
    bs, c, sh, sw = latent.shape
    if c != 48:
        raise ValueError('PADC requires 48 latent channels')
    jscc.decoder.enc_shape = [c, sh, sw]
    out = masked_channel(latent.flatten(1), n_active, snr, mode, power)
    return jscc.decoder(out, snr.reshape(bs, 1))


def jscc_forward(jscc, x, snr, rate, mode='awgn', return_latent=False):
    bs = x.size(0)
    if x.size(2) % 4 or x.size(3) % 4:
        raise ValueError('Image H and W must be divisible by 4')
    snr = torch.as_tensor(snr, device=x.device, dtype=torch.float32).reshape(-1, 1)
    if snr.size(0) == 1 and bs > 1:
        snr = snr.expand(bs, 1)
    n_active = channel_count(rate, bs, x.device)
    z0 = encode_features(jscc, x, snr)
    rec = reconstruct_from_latent(jscc, z0, snr, n_active, mode)
    if return_latent:
        return rec, z0, n_active
    return rec


def psnr_torch(original, decoded, quantized=True):
    """Per-image RGB PSNR, optionally mirroring author's uint8 truncation.

    The original utils.Img_transform uses float*255 cast to uint8, then
    scikit-image peak_signal_noise_ratio. No averaging MSE across images.
    """
    if quantized:
        a = (original.detach().clamp(0, 1) * 255).to(torch.uint8).float()
        b = (decoded.detach().clamp(0, 1) * 255).to(torch.uint8).float()
        mse = ((a-b).square()).mean(dim=(1, 2, 3)).clamp_min(1e-12)
        return 10.0 * torch.log10((255.0 ** 2) / mse)
    mse = (original.float()-decoded.float()).square().mean(dim=(1,2,3)).clamp_min(1e-12)
    return -10.0 * torch.log10(mse)


def hash_file(filepath, chunk=1024*1024):
    h=hashlib.sha256()
    with open(filepath,'rb') as fp:
        while True:
            b=fp.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()


@contextmanager
def torch_fixed_rng(seed, device):
    gpus = [torch.cuda.current_device()] if torch.device(device).type=='cuda' else []
    with torch.random.fork_rng(devices=gpus):
        torch.manual_seed(int(seed))
        if gpus: torch.cuda.manual_seed_all(int(seed))
        yield


def safe_torch_load(path, map_location='cpu'):
    """ONLY for checkpoints generated locally by this notebook; not arbitrary files."""
    return torch.load(path, map_location=map_location, weights_only=False)


def atomic_torch_save(obj, path):
    path=Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp=path.with_suffix(path.suffix+'.tmp')
    torch.save(obj, temp)
    os.replace(temp,path)


### Module: `padc_data.py`

In [ ]:
"""Dataset discovery and loaders: CIFAR10 train/val, CIFAR100 held-out, Kodak24 held-out."""
import re
import os
import zipfile
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, Subset
from PIL import Image


class NumpyCifar(Dataset):
    def __init__(self, path):
        self.path = Path(path)
        self.a = np.load(self.path, mmap_mode='r', allow_pickle=False)
        if self.a.ndim != 4:
            raise ValueError(f'Expected N x H x W x C or N x C x H x W in {path}, got {self.a.shape}')
        if self.a.shape[1] != 3 and self.a.shape[-1] != 3:
            raise ValueError(f'Missing RGB axis in {path}: {self.a.shape}')
    def __len__(self): return len(self.a)
    def __getitem__(self, i):
        a = np.array(self.a[i], copy=True)
        if a.shape[-1] == 3:
            a = np.transpose(a, (2, 0, 1))
        t = torch.from_numpy(np.ascontiguousarray(a)).float()
        if t.max() > 1.001: t = t / 255.0
        return t.clamp(0, 1)


class TorchvisionCifar(Dataset):
    def __init__(self, dataset):
        from torchvision import transforms
        self.dataset = dataset
        self.totensor = transforms.ToTensor()
    def __len__(self): return len(self.dataset)
    def __getitem__(self,i):
        img,_ = self.dataset[i]
        return self.totensor(img)


def cifar_dataset(name, split, root, npy_file=None, download=False):
    """Supports explicit x_train/x_test .npy OR torchvision's unpacked/downloaded CIFAR."""
    name=name.lower()
    if name not in ('cifar10','cifar100') or split not in ('train','test'):
        raise ValueError((name,split))
    root=Path(root).expanduser()
    if npy_file:
        path=Path(npy_file).expanduser()
        if not path.exists(): raise FileNotFoundError(f'Your CIFAR .npy path does not exist: {path}')
        return NumpyCifar(path)
    # Original repo uses these directories; if present select automatically.
    npy_dir = 'CIFAR10_raw' if name == 'cifar10' else 'CIFAR100_raw'
    for path in [root / npy_dir / f'x_{split}.npy', root / 'data' / npy_dir / f'x_{split}.npy', root / f'x_{split}.npy']:
        if path.is_file():
            print('Using CIFAR numpy:',path)
            return NumpyCifar(path)
    try:
        from torchvision.datasets import CIFAR10, CIFAR100
        dataset_type = CIFAR10 if name == 'cifar10' else CIFAR100
        root.mkdir(parents=True, exist_ok=True)
        ds = dataset_type(root=str(root), train=(split=='train'), download=download)
        return TorchvisionCifar(ds)
    except Exception as exc:
        raise RuntimeError(
            f'{name}/{split} not found in {root}. Set the explicit x_{split}.npy path, '
            f'point root at a torchvision CIFAR dataset, or enable download=True. Original error: {exc}'
        ) from exc


def dataloader(dataset, batch=64, shuffle=False, workers=2, seed=2026):
    g = torch.Generator().manual_seed(int(seed))
    return DataLoader(dataset, batch_size=int(batch), shuffle=shuffle,
                      num_workers=int(workers), pin_memory=torch.cuda.is_available(),
                      persistent_workers=workers>0, drop_last=False, generator=g)


def subset(dataset, max_count=None, seed=2026):
    if max_count is None or int(max_count) >= len(dataset):
        return dataset
    ids = np.random.default_rng(int(seed)).permutation(len(dataset))[:int(max_count)].tolist()
    return Subset(dataset, ids)


def resolve_kodak(root, extracted_dir=None, require_24=True):
    """Find kodim01.png ... kodim24.png recursively or unzip safely to temp dir."""
    root=Path(root).expanduser()
    if not root.exists(): raise FileNotFoundError(f'Kodak path does not exist: {root}')
    if root.is_file() and root.suffix.lower()=='.zip':
        if not extracted_dir: raise ValueError('Pass extracted_dir when Kodak is a zip file')
        dst=Path(extracted_dir)
        dst.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(root) as z:
            for item in z.infolist():
                target=(dst / item.filename).resolve()
                if not target.is_relative_to(dst.resolve()):
                    raise ValueError('Unsafe zip archive path')
            z.extractall(dst)
        root=dst
    if not root.is_dir(): raise NotADirectoryError(root)
    found={}
    for path in root.rglob('*'):
        if path.is_file() and re.fullmatch(r'kodim(0[1-9]|1[0-9]|2[0-4])\.(png|jpg|jpeg)', path.name, re.I):
            key=int(re.search(r'(\d\d)',path.stem).group(1))
            if key in found: raise ValueError(f'Duplicate Kodak #{key}: {path} vs {found[key]}')
            found[key]=path
    missing=[i for i in range(1,25) if i not in found]
    if require_24 and missing:
        raise FileNotFoundError(f'Kodak24 incomplete. Missing: {missing}, scanned: {root}')
    if not found: raise FileNotFoundError('No kodimXX.(png/jpg) images found in '+str(root))
    return [found[i] for i in sorted(found)]


class Kodak24(Dataset):
    def __init__(self, paths):
        self.paths=list(paths)
    def __len__(self): return len(self.paths)
    def __getitem__(self,i):
        with Image.open(self.paths[i]) as im:
            a=np.asarray(im.convert('RGB'), dtype=np.uint8).copy()
        x=torch.from_numpy(a).permute(2,0,1).float()/255.0
        h,w=x.shape[-2:]
        if h%4 or w%4: raise ValueError(f'Image not divisible by 4: {self.paths[i]} ({w}x{h})')
        if ((h//4)*(w//4))%2: raise ValueError(f'Complex pairing requires even latent area: {self.paths[i]}')
        return x, self.paths[i].stem


class ImageFolderPatches(Dataset):
    """Optional paper ImageNet phase: any recursive directory of real photos.

    This is intentionally not the Kodak test dataset. Kodak is never used here.
    """
    def __init__(self, paths, size=128, flip=False):
        from torchvision import transforms
        self.paths=list(paths)
        tr=[transforms.CenterCrop((size,size))]
        if flip:
            tr += [transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip()]
        tr += [transforms.ToTensor()]
        self.transform=transforms.Compose(tr)
    def __len__(self): return len(self.paths)
    def __getitem__(self,i):
        with Image.open(self.paths[i]) as im:
            x=self.transform(im.convert('RGB'))
        return x


def highres_dataloaders(directory, batch=32, workers=2, size=128, val_frac=0.1, seed=2026, limit=None):
    directory=Path(directory).expanduser()
    if not directory.is_dir(): raise FileNotFoundError(directory)
    exts={'.png','.jpeg','.jpg','.webp','.bmp'}
    paths=[p for p in directory.rglob('*') if p.suffix.lower() in exts and p.is_file()]
    # Exclude Kodak by filename and encourage explicit separation of training/test.
    paths=[p for p in paths if not re.fullmatch(r'kodim\d\d', p.stem, re.I)]
    if len(paths)<20: raise ValueError(f'Need >=20 high-resolution training pictures, got {len(paths)}')
    rng=np.random.default_rng(seed)
    paths=[paths[i] for i in rng.permutation(len(paths))]
    if limit is not None: paths=paths[:int(limit)]
    nval=max(1,int(len(paths)*val_frac))
    val, train=paths[:nval],paths[nval:]
    assert not set(val)&set(train)
    print(f'ImageNet-style patches: {len(train)} train, {len(val)} validation; no Kodak names included')
    return (dataloader(ImageFolderPatches(train,size,True),batch,True,workers,seed),
            dataloader(ImageFolderPatches(val,size,False),batch,False,workers,seed))


### Module: `padc_train.py`

In [ ]:
"""Checkpointable DeepJSCC-V learning on CIFAR10 and optional ImageNet-style crops."""
import csv
import time
from pathlib import Path
from tqdm.auto import tqdm
import torch
from torch import nn


def sample_train_conditions(batch, device, snr_low=0., snr_high=27., r_low=.05, r_high=.5):
    # Original GitHub uses integer SNR in {0,1,...,27}; physical rate is U[.05,.5].
    snr=torch.randint(int(snr_low), int(snr_high)+1, (batch,1), device=device).float()
    rate=r_low+(r_high-r_low)*torch.rand(batch,1,device=device)
    return snr,rate


@torch.no_grad()
def validate_jscc(jscc, loader, device, mode='awgn', max_batches=None, seed=5252):
    jscc.eval()
    numerator=0.
    denominator=0
    with torch_fixed_rng(seed, device):
        for j, x in enumerate(loader):
            if max_batches is not None and j >= int(max_batches): break
            if isinstance(x,(list,tuple)): x=x[0]
            x=x.to(device,non_blocking=True)
            snr,rate=sample_train_conditions(x.size(0),device)
            out=jscc_forward(jscc,x,snr,rate,mode)
            numerator += torch.nn.functional.mse_loss(out, x, reduction='sum').item()
            denominator += x.numel()
    if denominator==0: raise ValueError('No validation samples')
    return numerator / denominator


def _append_csv(path, row):
    path=Path(path)
    path.parent.mkdir(exist_ok=True,parents=True)
    fields=list(row)
    with open(path,'a',newline='',encoding='utf-8') as f:
        wr=csv.DictWriter(f,fieldnames=fields)
        if f.tell()==0: wr.writeheader()
        wr.writerow(row)


def fit_jscc(jscc, train_loader, val_loader, exp_dir, device, mode='awgn',
             epochs=400, lr=1e-4, resume=True, max_train_batches=None, max_val_batches=None,
             run_tag='cifar10', clip_grad=None, seed=2026):
    """Checkpoint last and best to Drive after each epoch. 400 epochs matches paper.
    Resume uses own optimizer state, explicit channel+shape validation.
    """
    exp_dir=Path(exp_dir)
    exp_dir.mkdir(exist_ok=True,parents=True)
    basename=f'jscc_v_{mode}_{run_tag}'
    best_path=exp_dir/f'{basename}_best.pt'
    last_path=exp_dir/f'{basename}_last.pt'
    csv_path=exp_dir/f'{basename}_history.csv'
    opt=torch.optim.Adam(jscc.parameters(), lr=float(lr))
    best_mse=float('inf'); start_epoch=0
    if resume and last_path.exists():
        ck=safe_torch_load(last_path, map_location=device)
        if ck['channel']!=mode or ck['run_tag']!=run_tag:
            raise RuntimeError('Wrong checkpoint provenance')
        jscc.load_state_dict(ck['state_dict'],strict=True)
        opt.load_state_dict(ck['optimizer'])
        best_mse=float(ck['best_mse']); start_epoch=int(ck['epoch'])+1
        print(f'Resuming {basename} from epoch {start_epoch}/{epochs}')
    elif resume and best_path.exists():
        print('Best weights exist but no last checkpoint; starting new training. '
              'Set resume=False to intentionally start from scratch.')
    if start_epoch >= epochs:
        print('Training already completed:',last_path)
        if best_path.exists(): jscc.load_state_dict(safe_torch_load(best_path,device)['state_dict'])
        return {'best':str(best_path),'last':str(last_path),'best_mse':best_mse,'epoch':start_epoch-1}
    for epoch in range(start_epoch,int(epochs)):
        since=time.time()
        jscc.train()
        total=0.; seen=0
        bar=tqdm(train_loader,desc=f'{run_tag}/{mode} JSCC {epoch+1}/{epochs}',leave=False)
        for batch_id, x in enumerate(bar):
            if max_train_batches is not None and batch_id>=int(max_train_batches):break
            if isinstance(x,(list,tuple)):x=x[0]
            x=x.to(device,non_blocking=True)
            snr,rate=sample_train_conditions(x.size(0),device)
            out=jscc_forward(jscc,x,snr,rate,mode)
            loss=torch.nn.functional.mse_loss(out,x)
            if not torch.isfinite(loss):
                raise FloatingPointError('Non-finite DeepJSCC loss; check fading and learning rate')
            opt.zero_grad(set_to_none=True)
            loss.backward()
            if clip_grad is not None: nn.utils.clip_grad_norm_(jscc.parameters(),float(clip_grad))
            opt.step()
            total+=loss.item()*x.size(0);seen+=x.size(0)
            if batch_id%25==0:bar.set_postfix(mse=round(total/max(seen,1),6))
        if not seen: raise RuntimeError('Training loader is empty')
        train_mse=total/seen
        val_mse=validate_jscc(jscc,val_loader,device,mode,max_val_batches,seed=seed+900)
        improved=val_mse<best_mse
        if improved:best_mse=val_mse
        record={'epoch':epoch+1,'train_mse':train_mse,'val_mse':val_mse,
                'best_val_mse':best_mse,'elapsed_seconds':round(time.time()-since,1)}
        _append_csv(csv_path,record)
        payload={'state_dict':jscc.state_dict(),'optimizer':opt.state_dict(),
                 'epoch':epoch,'epochs_total':int(epochs),'best_mse':best_mse,
                 'channel':mode,'run_tag':run_tag,'kernel':5,'latent_channels':48,
                 'encoder_width':next((m.out_channels for m in jscc.encoder.modules() if isinstance(m,nn.Conv2d)),None),
                 'paper_cr_range':[.05,.5],'paper_snr_range':[0.,27.]}
        atomic_torch_save(payload,last_path)
        if improved:atomic_torch_save(payload,best_path)
        print(f'{run_tag}/{mode}: {epoch+1}/{epochs}: train={train_mse:.6f} '
              f'val={val_mse:.6f} best={best_mse:.6f} sec={record["elapsed_seconds"]} '
              f'{"[BEST]" if improved else ""}')
    if best_path.exists():jscc.load_state_dict(safe_torch_load(best_path,device)['state_dict'])
    jscc.eval()
    return {'best':str(best_path),'last':str(last_path),'best_mse':best_mse,'epoch':int(epochs)-1}


def load_jscc(jscc,best_path,device,mode=None):
    if not Path(best_path).exists():
        raise FileNotFoundError(f'No JSCC checkpoint: {best_path}; run the JSCC training cell first')
    ck=safe_torch_load(best_path, map_location=device)
    if mode is not None and ck['channel']!=mode:
        raise ValueError(f'Model was trained on {ck["channel"]}, requested {mode}')
    jscc.load_state_dict(ck['state_dict'],strict=True)
    jscc.eval()
    return ck


### Module: `padc_oracle.py`

In [ ]:
"""Generation of faithful OraNet teacher targets, training and validation."""
import json
import time
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from tqdm.auto import tqdm


@torch.no_grad()
def build_teacher_table(jscc, loader, device, mode, output_path,
                        draws_per_image=4, channel_repeats=1, max_batches=None,
                        seed=12345, jscc_sha=None, overwrite=False):
    """Generate (CWM, CWSTD, SNR, n_active) -> PSNR pairs from frozen DeepJSCC-V.

    Repeating draws samples multiple SNR/rates for *each* original image. A
    fresh channel realization is used on each repetition. Image-level PSNR
    targets are the mean dB across channel_repeats, not PSNR of mean image.
    """
    output_path=Path(output_path)
    output_path.parent.mkdir(exist_ok=True,parents=True)
    if output_path.exists() and not overwrite:
        with np.load(output_path,allow_pickle=False) as obj:
            assert obj['X'].shape[1]==98 and len(obj['X'])==len(obj['y'])
            meta=json.loads(str(obj['metadata']))
            required={'draws_per_image':int(draws_per_image),
                      'channel_repeats':int(channel_repeats),
                      'mode':mode,'seed':int(seed),
                      'teacher_weights_sha256':jscc_sha}
            if any(meta.get(k)!=v for k,v in required.items()):
                raise ValueError('Stale OraNet label-cache configuration. Delete cache or set overwrite=True.')
            print('Reusing teacher cache:',output_path, 'rows:',len(obj['y']))
        return str(output_path)
    if draws_per_image<1 or channel_repeats<1: raise ValueError('Need positive sampling counts')
    jscc.eval()
    all_X=[]; all_Y=[]; nimgs=0
    with torch_fixed_rng(seed,device):
        for bid,x in enumerate(tqdm(loader,desc=f'Teacher labels ({mode})',leave=False)):
            if max_batches is not None and bid>=int(max_batches):break
            if isinstance(x,(list,tuple)):x=x[0]
            x=x.to(device,non_blocking=True)
            nimgs+=x.size(0)
            for _ in range(int(draws_per_image)):
                snr,rate=sample_train_conditions(x.size(0),device)
                active=channel_count(rate,x.size(0),device)
                latent=encode_features(jscc,x,snr)
                features=feature_vector(latent,snr,active)
                yy=[]
                for _ in range(int(channel_repeats)):
                    rec=reconstruct_from_latent(jscc,latent,snr,active,mode)
                    yy.append(psnr_torch(x,rec,quantized=True))
                y=torch.stack(yy).mean(0)
                if not torch.isfinite(y).all():
                    raise FloatingPointError('Non-finite teacher PSNR')
                all_X.append(features.detach().cpu().numpy())
                all_Y.append(y.detach().cpu().numpy())
    if not all_X:raise RuntimeError('No teacher samples')
    X=np.concatenate(all_X).astype('float32')
    Y=np.concatenate(all_Y).astype('float32')
    np.savez_compressed(output_path,X=X,y=Y,
                        metadata=json.dumps({'samples':len(Y),'images':nimgs,
                                             'draws_per_image':int(draws_per_image),
                                             'channel_repeats':int(channel_repeats),
                                             'mode':mode,'seed':int(seed),
                                             'teacher_weights_sha256':jscc_sha,
                                             'label_metric':'uint8_psnr_after_truncation'}))
    print('Teacher cache:',output_path,'rows:',len(Y),'label dB range:',(float(Y.min()),float(Y.max())))
    return str(output_path)


def load_teacher(path):
    with np.load(path, allow_pickle=False) as f:
        X=torch.from_numpy(f['X'].copy()).float()
        y=torch.from_numpy(f['y'].copy()).float()
        meta=json.loads(str(f['metadata']))
    if X.ndim!=2 or X.shape[1]!=98 or y.ndim!=1 or len(X)!=len(y):
        raise ValueError('Bad OraNet table schema')
    if not torch.isfinite(X).all() or not torch.isfinite(y).all():
        raise ValueError('Non-finite Oracle data')
    return X,y,meta


@torch.no_grad()
def validate_oracle(oracle,X,y,device,batch=8192):
    oracle.eval()
    preds=[]
    for i in range(0,len(X),batch):
        pred=oracle_forward(oracle,X[i:i+batch].to(device))
        preds.append(pred.cpu())
    p=torch.cat(preds)
    residual=p-y
    return {'mae_db':float(residual.abs().mean()),
            'rmse_db':float(residual.square().mean().sqrt()),
            'mse_db2':float(residual.square().mean()),
            'mean_signed_error_db':float(residual.mean()),
            'q95_overprediction_db':float(torch.quantile(residual,0.95)),
            'p':p,'y':y.clone()}


def fit_oracle_cached(oracle, train_table, val_table, output_dir, device, mode,
                      epochs=150,lr=1e-4,batch=512,resume=True,run_tag='cifar10'):
    """Cached-label training: SAME regression objective as paper, cheaper teacher inference.
    For a strict online teacher variant see fit_oracle_online().
    """
    trainX,trainY,train_meta=load_teacher(train_table)
    valX,valY,val_meta=load_teacher(val_table)
    if train_meta['mode']!=mode or val_meta['mode']!=mode:raise ValueError('Oracle teacher channel mismatch')
    if train_meta['teacher_weights_sha256']!=val_meta['teacher_weights_sha256']:
        raise ValueError('Oracle train/val teacher networks differ!')
    td=TensorDataset(trainX,trainY)
    train_loader=DataLoader(td,batch_size=int(batch),shuffle=True,num_workers=0)
    return _fit_oracle_common(oracle,train_loader,valX,valY,output_dir,device,mode,
                              epochs,lr,resume,run_tag,train_meta['teacher_weights_sha256'],
                              live_jscc=None,live_loader=None)


def fit_oracle_online(oracle, jscc, train_loader, val_table, output_dir, device, mode,
                      teacher_sha,epochs=150,lr=1e-4,resume=True,run_tag='cifar10'):
    """Paper-like training: regenerate SNR, CR and stochastic PSNR every epoch.
    This is much more GPU-expensive than the recommended cached teacher option.
    """
    valX,valY,val_meta=load_teacher(val_table)
    if val_meta['mode']!=mode or val_meta['teacher_weights_sha256']!=teacher_sha:
        raise ValueError('Wrong validation table teacher')
    jscc.eval()
    return _fit_oracle_common(oracle,None,valX,valY,output_dir,device,mode,
                              epochs,lr,resume,run_tag,teacher_sha,
                              live_jscc=jscc,live_loader=train_loader)


def _fit_oracle_common(oracle,train_table_loader,valX,valY,output_dir,device,mode,
                       epochs,lr,resume,run_tag,teacher_sha,live_jscc=None,live_loader=None):
    output_dir=Path(output_dir);output_dir.mkdir(parents=True,exist_ok=True)
    basename=f'oracle_{mode}_{run_tag}'
    best=output_dir/f'{basename}_best.pt'
    last=output_dir/f'{basename}_last.pt'
    history=output_dir/f'{basename}_history.csv'
    opt=torch.optim.Adam(oracle.parameters(),lr=float(lr))
    bestmse=float('inf');start=0
    if resume and last.exists():
        ck=safe_torch_load(last,device)
        if ck['channel']!=mode or ck['teacher_sha']!=teacher_sha or ck['run_tag']!=run_tag:
            raise ValueError('Existing OraNet checkpoint belongs to another DeepJSCC model or dataset')
        oracle.load_state_dict(ck['state_dict']);opt.load_state_dict(ck['optimizer'])
        bestmse=float(ck['best_mse']);start=int(ck['epoch'])+1
        print('Resuming OraNet at epoch',start)
    if start>=int(epochs):
        if best.exists():oracle.load_state_dict(safe_torch_load(best,device)['state_dict'])
        return {'best':str(best),'last':str(last),'epoch':start-1,'best_val_mse':bestmse}
    if start == 0:
        # Original OraNet ends in ReLU; random negative head biases can leave
        # all predictions at zero and kill gradients. A positive initial output
        # bias changes only initialization, not the author's architecture.
        # If you want the exact upstream random initialization, comment this.
        with torch.no_grad():
            nn_head=oracle.fc3
            nn_head.weight.zero_()
            nn_head.bias.fill_(25.0)
        print('OraNet PSNR head initialized to 25 dB to avoid final-ReLU dead neurons.')
    for epoch in range(start,int(epochs)):
        since=time.time();oracle.train()
        count=0;total=0.
        iterator=train_table_loader if live_jscc is None else live_loader
        for bid, sample in enumerate(tqdm(iterator,desc=f'OraNet {mode} {epoch+1}/{epochs}',leave=False)):
            if live_jscc is None:
                feat,y=sample
                feat=feat.to(device,non_blocking=True)
                y=y.to(device,non_blocking=True)
            else:
                x=sample[0] if isinstance(sample,(list,tuple)) else sample
                x=x.to(device,non_blocking=True)
                with torch.no_grad():
                    snr,r=sample_train_conditions(x.size(0),device)
                    nch=channel_count(r,x.size(0),device)
                    z0=encode_features(live_jscc,x,snr)
                    rec=reconstruct_from_latent(live_jscc,z0,snr,nch,mode)
                    feat=feature_vector(z0,snr,nch).detach()
                    y=psnr_torch(x,rec,quantized=True).detach()
            pred=oracle_forward(oracle,feat)
            loss=torch.nn.functional.mse_loss(pred,y)
            if not torch.isfinite(loss):raise FloatingPointError('OraNet training loss not finite')
            opt.zero_grad(set_to_none=True);loss.backward();opt.step()
            count+=len(y);total+=float(loss.item())*len(y)
        if not count:raise RuntimeError('No OraNet training pairs')
        metrics=validate_oracle(oracle,valX,valY,device)
        valmse=metrics['mse_db2'];improved=valmse<bestmse
        if improved:bestmse=valmse
        payload={'state_dict':oracle.state_dict(),'optimizer':opt.state_dict(),
                 'epoch':epoch,'epochs_total':int(epochs),'best_mse':bestmse,
                 'channel':mode,'run_tag':run_tag,'teacher_sha':teacher_sha,
                 'features':['mu48','std48','snr_db','active_channels'],
                 'label':'uint8_per_image_PSNR_dB'}
        atomic_torch_save(payload,last)
        if improved:atomic_torch_save(payload,best)
        _append_csv(history,{'epoch':epoch+1,'train_mse_db2':total/count,'val_mse_db2':valmse,
                             'val_mae_db':metrics['mae_db'],'best_mse_db2':bestmse,
                             'seconds':round(time.time()-since,1)})
        print(f'OraNet {epoch+1}/{epochs}: train mse={total/count:.4f} '
              f'val mse={valmse:.4f} val MAE={metrics["mae_db"]:.3f} dB '
              f'{"[BEST]" if improved else ""}')
    oracle.load_state_dict(safe_torch_load(best,device)['state_dict'])
    oracle.eval()
    return {'best':str(best),'last':str(last),'epoch':int(epochs)-1,'best_val_mse':bestmse}


def load_oracle(oracle,path,device,mode,teacher_sha):
    if not Path(path).exists():raise FileNotFoundError(path)
    ck=safe_torch_load(path,device)
    if ck['channel']!=mode or ck['teacher_sha']!=teacher_sha:
        raise ValueError('OraNet is specific to one DeepJSCC checkpoint AND channel, as paper requires')
    oracle.load_state_dict(ck['state_dict'],strict=True)
    oracle.eval()
    return ck


### Module: `padc_eval.py`

In [ ]:
"""Out-of-sample rate/PSNR evaluation and exhaustive PADC CR optimization."""
import json
from pathlib import Path
import math
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm


def _get_batch_x_and_names(sample, num_seen=0):
    if isinstance(sample,(list,tuple)):
        if len(sample)==2 and isinstance(sample[1],(list,tuple)) and len(sample[1])>0 and isinstance(sample[1][0],str):
            x=sample[0]
            return x, list(sample[1])
        x=sample[0]
    else:
        x=sample
    return x, [f'img_{num_seen+i:05d}' for i in range(x.size(0))]


def canonicalize_rates(rate_grid):
    """Deduplicate paper nominal CRs mapping to the same 48-channel mask.

    Example: 1/19 and 1/20 both keep five channels at some quantized rates.
    """
    out=[];seen=set()
    for r in sorted(float(v) for v in rate_grid):
        c=int(channel_count(r).item())
        if c not in seen:
            seen.add(c);out.append(r)
    if len(out)<len(rate_grid):
        print(f'Quantized effective CRs: {len(rate_grid)} nominal -> {len(out)} distinct 48-channel masks.')
    return out


def _as_batch_loader(dataset, batch, device='cpu',workers=0):
    return DataLoader(dataset,batch_size=int(batch),shuffle=False,num_workers=workers,
                      pin_memory=torch.device(device).type=='cuda',drop_last=False)


@torch.no_grad()
def evaluate_grid(jscc, oracle, dataset, dataset_name, snr_grid, rate_grid,
                  device, mode='awgn', mc_repeats=1, max_images=None,
                  batch=64, output_csv=None, seed=2026, resume=True, model_sha=None, oracle_sha=None):
    """Mean per-image PSNR at a complete fixed SNR x CR grid.

    Incrementally appends *complete batches* to CSV to survive Colab timeouts.
    Resuming checks settings and completed image IDs. Use model_sha to prevent
    accidentally mixing measurements from two different pretrained networks.
    """
    jscc.eval()
    if oracle is not None: oracle.eval()
    raw_rate_grid=list(rate_grid)
    rate_grid=canonicalize_rates(raw_rate_grid)
    done=0
    if output_csv is not None:
        output_csv=Path(output_csv)
        output_csv.parent.mkdir(parents=True,exist_ok=True)
        meta_file=output_csv.with_suffix('.meta.json')
        signature={'dataset_name':dataset_name,'channel':mode,
                   'snr_db':list(map(float,snr_grid)),
                   'cr':list(map(float,rate_grid)),
                   'nominal_cr':list(map(float,raw_rate_grid)),
                   'mc_repeats':int(mc_repeats),'max_images':max_images,
                   'seed':int(seed),'model_sha256':model_sha,
                   'psnr_metric':'uint8_truncation'}
        if oracle_sha is not None:
            signature['oracle_sha256'] = oracle_sha
        if output_csv.exists() and resume:
            if not meta_file.exists():
                raise RuntimeError('Existing CSV lacks experiment metadata; rename or delete it before running')
            previous=json.loads(meta_file.read_text())
            if previous != signature:
                raise ValueError('Existing evaluation CSV settings/weights are DIFFERENT. Use a new output path.')
            try:
                previous_rows=pd.read_csv(output_csv)
            except Exception as exc:
                raise RuntimeError('Incomplete CSV write detected. Inspect/repair the tail before resuming.') from exc
            n_expected=len(snr_grid)*len(rate_grid)
            completed=set(previous_rows.groupby('image_id').size().loc[lambda s:s==n_expected].index.tolist())
            print(f'Resuming evaluation: {len(completed)} complete image(s) already saved')
        else:
            completed=set()
            if output_csv.exists():output_csv.unlink()
            meta_file.write_text(json.dumps(signature,indent=2),encoding='utf8')
    else:
        completed=set();all_rows=[]
    loader=_as_batch_loader(dataset,batch,device)
    with torch_fixed_rng(seed,device):
        for sample in tqdm(loader,desc=f'Grid {dataset_name} / {mode}',leave=False):
            x,names=_get_batch_x_and_names(sample,done)
            if max_images is not None and done>=int(max_images):break
            if max_images is not None and done+x.size(0)>int(max_images):
                n=int(max_images)-done;x=x[:n];names=names[:n]
            done+=len(names)
            if all(name in completed for name in names):continue
            indices=[i for i,name in enumerate(names) if name not in completed]
            names=[names[i] for i in indices]
            x=x[indices].to(device,non_blocking=True)
            if x.size(2)%4 or x.size(3)%4:
                raise ValueError('Input height/width must be divisible by 4')
            rows=[]
            for snr_value in snr_grid:
                snr=torch.full((x.size(0),1),float(snr_value),device=device)
                latent=encode_features(jscc,x,snr)
                for paper_r in rate_grid:
                    nactive=channel_count(float(paper_r),len(x),device)
                    feats=feature_vector(latent,snr,nactive)
                    pred=oracle_forward(oracle,feats) if oracle is not None else torch.full((len(x),),float('nan'),device=device)
                    vals=[]
                    for _ in range(int(mc_repeats)):
                        rec=reconstruct_from_latent(jscc,latent,snr,nactive,mode)
                        vals.append(psnr_torch(x,rec,quantized=True))
                    vals=torch.stack(vals)
                    av=vals.mean(0).cpu().numpy()
                    std=vals.std(0,unbiased=False).cpu().numpy()
                    predictions=pred.cpu().numpy()
                    active=nactive.reshape(-1).cpu().numpy()
                    for i,name in enumerate(names):
                        rows.append({'dataset':dataset_name,'image_id':str(name),
                                     'snr_db':float(snr_value),'requested_r':float(paper_r),
                                     'active_channels':int(active[i]),'effective_r':float(active[i])/96.,
                                     'psnr_db':float(av[i]),'channel_psnr_std_db':float(std[i]),
                                     'predicted_psnr_db':float(predictions[i]),
                                     'channel':mode,'channel_repeats':int(mc_repeats)})
            batch_df=pd.DataFrame(rows)
            if output_csv is None:
                all_rows.extend(rows)
            else:
                batch_df.to_csv(output_csv,index=False,mode='a',header=not output_csv.exists())
    if output_csv is None:
        if not all_rows:raise RuntimeError('Empty dataset')
        return pd.DataFrame(all_rows)
    if not output_csv.exists():
        raise RuntimeError('No rows were generated and no old CSV exists')
    df=pd.read_csv(output_csv)
    # Unique complete image/SNR/rate combinations are required for fair optimization.
    keys=['image_id','snr_db','active_channels']
    if df.duplicated(keys).any():
        raise ValueError('Duplicate evaluation rows detected; reset this CSV and rerun')
    print('Evaluation saved:',output_csv,'images:',df.image_id.nunique(),'rows:',len(df))
    return df

def summarize_grid(df):
    result=(df.groupby(['dataset','channel','snr_db','requested_r','effective_r'],as_index=False)
              .agg(mean_psnr_db=('psnr_db','mean'),
                   mean_pred_psnr_db=('predicted_psnr_db','mean'),
                   mae_db=('psnr_db',lambda z: float('nan')),
                   number_of_images=('image_id','size')))
    # MAE of single-image predictions (not difference of global means).
    p=df.copy()
    p['ae']=(p['predicted_psnr_db']-p['psnr_db']).abs()
    mae=(p.groupby(['dataset','channel','snr_db','requested_r','effective_r'],as_index=False)['ae']
           .mean().rename(columns={'ae':'mae_db'}))
    result=result.drop(columns=['mae_db']).merge(mae,how='left',on=['dataset','channel','snr_db','requested_r','effective_r'])
    return result.sort_values(['dataset','snr_db','requested_r'])


def lookup_data_level(validation_grid, thresholds, snr_grid=None):
    """Select ONE CR for all images using only CIFAR10 validation average PSNR.

    Requiring mean PSNR is the paper's *data-level* problem, NOT a per-image
    guarantee. Returns infeasibility flags for targets above max supported CR.
    """
    if snr_grid is None:snr_grid=sorted(validation_grid['snr_db'].unique())
    av=(validation_grid.groupby(['snr_db','effective_r'],as_index=False)['psnr_db'].mean())
    rows=[]
    for snr in snr_grid:
        rr=av[np.isclose(av.snr_db,float(snr))].sort_values('effective_r')
        if rr.empty:raise ValueError(f'Missing SNR {snr} in validation table')
        for target in thresholds:
            okay=rr[rr['psnr_db']>=float(target)]
            feasible=len(okay)>0
            chosen=okay.iloc[0] if feasible else rr.iloc[-1]
            rows.append({'snr_db':float(snr),'target_psnr_db':float(target),
                         'selected_r':float(chosen['effective_r']),
                         'validation_expected_psnr_db':float(chosen['psnr_db']),
                         'predicted_feasible':bool(feasible),
                         'rule':'data_level_validation_mean'})
    return pd.DataFrame(rows)


def optimize_from_grid(df, targets, data_lookup=None, conservative_margin=0.):
    """Exhaustive minima from a measured complete grid.

    The ground-truth best CR is a POST-HOC benchmark using test PSNR and CANNOT
    be used by a real transmitter; PADC uses only the OraNet prediction.
    """
    margin=max(0.,float(conservative_margin))
    rows=[]
    for (dataset,ch,img,snr),table in df.groupby(['dataset','channel','image_id','snr_db'],sort=False):
        table=table.sort_values('effective_r').drop_duplicates('active_channels')
        for threshold in targets:
            gt=table[table['psnr_db']>=threshold]
            pred=table[table['predicted_psnr_db']-margin>=threshold]
            p_ok=len(pred)>0;t_ok=len(gt)>0
            chosen=pred.iloc[0] if p_ok else table.iloc[-1]
            oracle_opt=gt.iloc[0] if t_ok else table.iloc[-1]
            row={'dataset':dataset,'channel':ch,'image_id':img,
                 'snr_db':float(snr),'target_psnr_db':float(threshold),
                 'padc_r':float(chosen['effective_r']),
                 'padc_predicted_psnr_db':float(chosen['predicted_psnr_db']),
                 'padc_true_psnr_db':float(chosen['psnr_db']),
                 'padc_prediction_feasible':bool(p_ok),
                 'padc_test_satisfied':bool(chosen['psnr_db']>=threshold),
                 'genie_r':float(oracle_opt['effective_r']),
                 'genie_test_feasible':bool(t_ok),
                 # Comparison is only meaningful if the selected PADC rate
                 # actually satisfies the target and a feasible genie exists.
                 'excess_r_vs_genie':(float(chosen['effective_r']-oracle_opt['effective_r'])
                                      if bool(t_ok) and float(chosen['psnr_db'])>=threshold
                                      else float('nan')),
                 'calibration_margin_db':margin}
            if data_lookup is not None:
                match=data_lookup[np.isclose(data_lookup['snr_db'],float(snr)) &
                                  np.isclose(data_lookup['target_psnr_db'],float(threshold))]
                if len(match)!=1:raise ValueError(f'Missing data baseline SNR/target: {snr}/{threshold}')
                rate=float(match.iloc[0]['selected_r'])
                base=table[np.isclose(table['effective_r'],rate)]
                if len(base)!=1:raise ValueError(f'Data-level rate {rate} missing from grid')
                base=base.iloc[0]
                row.update(data_level_r=rate,
                           data_level_true_psnr_db=float(base['psnr_db']),
                           data_level_test_satisfied=bool(base['psnr_db']>=threshold))
            rows.append(row)
    return pd.DataFrame(rows)


def summarize_optimizer(df):
    cols={'padc_r':'mean_padc_r', 'genie_r':'mean_genie_r',
          'padc_prediction_feasible':'pred_feasibility_fraction',
          'padc_test_satisfied':'padc_success_fraction',
          'excess_r_vs_genie':'mean_excess_r_vs_genie'}
    if 'data_level_r' in df:
        cols.update(data_level_r='mean_data_level_r',
                    data_level_test_satisfied='data_success_fraction')
    x=(df.groupby(['dataset','channel','snr_db','target_psnr_db'],as_index=False)
          .agg(**{v:(k,'mean') for k,v in cols.items()}))
    return x


def empirical_q95_margin(valX,valY,oracle,device):
    """Empirical one-sided calibration on CIFAR10 held-out labels (no Kodak).

    It does not mathematically guarantee per-image coverage under domain shift.
    """
    metrics=validate_oracle(oracle,valX,valY,device)
    return max(0.,metrics['q95_overprediction_db']),metrics


@torch.no_grad()
def reconstruct_example(jscc,oracle,dataset,index,snr_db,rate,device,mode='awgn'):
    sample=dataset[index]
    x,name=(sample if isinstance(sample,tuple) and len(sample)==2 else (sample,f'image_{index}'))
    x=x.unsqueeze(0).to(device)
    snr=torch.tensor([[float(snr_db)]],device=device)
    nactive=channel_count(rate,1,device)
    z0=encode_features(jscc,x,snr)
    pred=oracle_forward(oracle,feature_vector(z0,snr,nactive)).item()
    out=reconstruct_from_latent(jscc,z0,snr,nactive,mode)
    true=psnr_torch(x,out).item()
    return {'name':str(name),'original':x[0].cpu(),'recovered':out[0].clamp(0,1).cpu(),
            'psnr_db':true,'predicted_db':pred,'effective_r':nactive.item()/96.}


def report_metrics(grid, opt=None):
    p=grid['psnr_db']; yhat=grid['predicted_psnr_db'];delta=yhat-p
    out={'evaluated_cases':len(grid),'unique_images':int(grid.image_id.nunique()),
         'mean_real_psnr_db':float(p.mean()),'oracle_mae_db':float(delta.abs().mean()),
         'oracle_mse_db2':float((delta*delta).mean()),
         'oracle_signed_bias_db':float(delta.mean())}
    if opt is not None:
        out.update({'padc_mean_r':float(opt['padc_r'].mean()),
                    'padc_constraint_success':float(opt['padc_test_satisfied'].mean()),
                    'padc_predicted_feasible_fraction':float(opt['padc_prediction_feasible'].mean()),
                    'mean_excess_r_vs_genie':float(opt['excess_r_vs_genie'].mean())})
        if 'data_level_r' in opt:
            out['data_level_mean_r']=float(opt['data_level_r'].mean())
            out['data_level_constraint_success']=float(opt['data_level_test_satisfied'].mean())
    return out


### Module: `padc_plot.py`

In [ ]:
"""Matplotlib figures corresponding to the main PADC paper comparisons."""
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt


def plot_psnr_grid(df, dataset, out_dir, title_suffix=''):
    sub=df[df['dataset']==dataset]
    if sub.empty:return
    fig,ax=plt.subplots(figsize=(8.5,5.2))
    for rate, grp in sub.groupby('effective_r'):
        ss=grp.groupby('snr_db',as_index=False).agg(real=('psnr_db','mean'),
             prediction=('predicted_psnr_db','mean'))
        p=ax.plot(ss['snr_db'],ss['real'],marker='o',label=f'R={rate:.4f} actual')[0]
        if ss['prediction'].notna().any():
            ax.plot(ss['snr_db'],ss['prediction'],linestyle='--',color=p.get_color(),alpha=.75,
                    label=f'R={rate:.4f} OraNet')
    ax.set_xlabel('Channel SNR (dB)');ax.set_ylabel('PSNR (dB)')
    ax.set_title(f'{dataset}: DeepJSCC-V and OraNet {title_suffix}')
    ax.grid(True,alpha=.3);ax.legend(ncol=2,fontsize=8);fig.tight_layout()
    Path(out_dir).mkdir(parents=True,exist_ok=True)
    path=Path(out_dir)/f'{dataset}_psnr_vs_snr.png'
    fig.savefig(path,dpi=180);plt.show();return path


def plot_oracle_error(df, dataset, out_dir):
    d=df[df['dataset']==dataset].copy()
    if d.empty:return
    d['err_sq']=(d['psnr_db']-d['predicted_psnr_db'])**2
    avg=d.groupby(['snr_db','effective_r'],as_index=False)['err_sq'].mean()
    fig,ax=plt.subplots(figsize=(8.0,4.8))
    for rate,grp in avg.groupby('effective_r'):
        ax.plot(grp.snr_db,grp.err_sq,marker='o',label=f'R={rate:.4f}')
    ax.set(xlabel='Channel SNR (dB)',ylabel='Mean squared PSNR prediction error (dB²)',
           title=f'{dataset}: OraNet prediction MSE')
    ax.grid(alpha=.3);ax.legend(fontsize=8,ncol=2);fig.tight_layout()
    path=Path(out_dir)/f'{dataset}_oranet_mse.png'
    fig.savefig(path,dpi=180);plt.show();return path


def plot_rate_selection(opt, dataset, out_dir, image=None, tag=""):
    d=opt[opt['dataset']==dataset]
    if image is not None:d=d[d['image_id']==image]
    if d.empty:return
    fig,ax=plt.subplots(figsize=(8,5))
    for th,grp in d.groupby('target_psnr_db'):
        g=grp.groupby('snr_db',as_index=False).mean(numeric_only=True)
        a=ax.plot(g.snr_db,g.padc_r,marker='o',label=f'PADC, {th:g} dB')[0]
        ax.plot(g.snr_db,g.genie_r,linestyle=':',color=a.get_color(),alpha=.8,
                label=f'Genie after test, {th:g} dB')
        if 'data_level_r' in g:
            ax.plot(g.snr_db,g.data_level_r,linestyle='--',alpha=.65,
                    label=f'Data-level, {th:g} dB')
    ax.set(xlabel='Channel SNR (dB)',ylabel='Selected physical CR (K/N)',
           title=f'{dataset} CR optimization'+(f' ({image})' if image else ''))
    ax.grid(alpha=.3);ax.legend(fontsize=8,ncol=2);fig.tight_layout()
    path=Path(out_dir)/(f'{dataset}_{image}{tag}_cr_selection.png' if image else f'{dataset}{tag}_cr_selection.png')
    fig.savefig(path,dpi=180);plt.show();return path


def plot_example(example,out_dir):
    orig=example['original'].permute(1,2,0).numpy()
    reco=example['recovered'].permute(1,2,0).numpy()
    fig,axes=plt.subplots(1,2,figsize=(11,4.5))
    axes[0].imshow(orig);axes[0].set_title('Source')
    axes[1].imshow(reco)
    axes[1].set_title(f'Reconstructed {example["psnr_db"]:.2f} dB | OraNet {example["predicted_db"]:.2f} dB')
    for ax in axes:ax.axis('off')
    fig.suptitle(f'{example["name"]}, R={example["effective_r"]:.4f}')
    fig.tight_layout()
    path=Path(out_dir)/(f'{example["name"]}_reconstruction.png')
    path.parent.mkdir(parents=True,exist_ok=True)
    fig.savefig(path,dpi=160);plt.show();return path


### Module: `padc_baseline.py`

In [ ]:
"""Optional fixed-rate author's ADJSCC for the Fig.4 same-architecture baseline."""
from pathlib import Path
import torch
from torch import nn
from tqdm.auto import tqdm


def create_adjscc_fixed(rate,n_conv=256,kernel_size=5,device='cpu',height=32,width=32):
    from models import ADJSCC
    patch_author_attention()
    active=int(channel_count(rate).item())
    if ((height//4)*(width//4)*active)%2:
        raise ValueError('Odd I/Q feature dimensions in ADJSCC')
    return ADJSCC([active,height//4,width//4],kernel_size,n_conv).to(device)


def fixed_rate_forward(model,x,snr,mode='awgn'):
    bs=x.size(0)
    snr=torch.as_tensor(snr,device=x.device,dtype=torch.float32).reshape(bs,1)
    z=model.encoder(x,snr)
    real_count=z.size(1)
    energy=z.float().square().sum(1,keepdim=True).clamp_min(1e-8)
    z=z*(real_count/energy).sqrt()
    if mode=='awgn':
        z=z+torch.randn_like(z)*torch.pow(10.,-snr/20.)
    elif mode in ('slow_rayleigh','repo_fading'):
        z=_complex_channel(z,snr,torch.ones_like(z),mode,1.)
    else:raise ValueError(mode)
    model.decoder.enc_shape=[model.decoder.enc_shape[0],x.size(-2)//4,x.size(-1)//4]
    return model.decoder(z,snr)


def fit_adjscc_fixed(model,rate,train_loader,val_loader,output_dir,device,mode='awgn',
                     epochs=200,lr=1e-4,resume=True,max_batches=None,max_val_batches=None):
    """Runs four independent baselines; NOT required for the OraNet/PADC pipeline."""
    c=int(channel_count(rate).item())
    prefix=f'adjscc_fixed_{mode}_c{c:02d}'
    output_dir=Path(output_dir);output_dir.mkdir(parents=True,exist_ok=True)
    path=output_dir/f'{prefix}_best.pt';last=output_dir/f'{prefix}_last.pt'
    opt=torch.optim.Adam(model.parameters(),lr=lr)
    start=0;best=float('inf')
    if resume and last.exists():
        ck=safe_torch_load(last,device)
        if ck['channel']!=mode or ck['nactive']!=c:raise ValueError('Baseline checkpoint mismatch')
        model.load_state_dict(ck['state_dict']);opt.load_state_dict(ck['optimizer'])
        start=ck['epoch']+1;best=ck['best_mse']
    for ep in range(start,int(epochs)):
        model.train();seen=0;total=0.
        for bid,x in enumerate(tqdm(train_loader,desc=f'ADJSCC {c}/48 ep{ep+1}',leave=False)):
            if max_batches is not None and bid>=max_batches:break
            if isinstance(x,(list,tuple)):x=x[0]
            x=x.to(device)
            snr=torch.randint(0,28,(x.size(0),1),device=device).float()
            out=fixed_rate_forward(model,x,snr,mode)
            loss=torch.nn.functional.mse_loss(out,x)
            opt.zero_grad(set_to_none=True);loss.backward();opt.step()
            total+=loss.item()*len(x);seen+=len(x)
        if not seen:raise RuntimeError('Empty ADJSCC train dataset')
        model.eval();v=0.;n=0
        with torch.no_grad(),torch_fixed_rng(5252,device):
            for bid,x in enumerate(val_loader):
                if max_val_batches is not None and bid>=max_val_batches:break
                if isinstance(x,(list,tuple)):x=x[0]
                x=x.to(device);snr=torch.randint(0,28,(x.size(0),1),device=device).float()
                out=fixed_rate_forward(model,x,snr,mode)
                v+=torch.nn.functional.mse_loss(out,x,reduction='sum').item();n+=x.numel()
        val_mse=v/n
        improved=val_mse<best
        if improved:best=val_mse
        save={'state_dict':model.state_dict(),'optimizer':opt.state_dict(),
              'epoch':ep,'best_mse':best,'nactive':c,'channel':mode}
        atomic_torch_save(save,last)
        if improved:atomic_torch_save(save,path)
        print(f'ADJSCC c={c} epoch {ep+1} val MSE={val_mse:.6f}')
    if path.exists():model.load_state_dict(safe_torch_load(path,device)['state_dict'])
    return str(path)


@torch.no_grad()
def evaluate_adjscc_fixed(model,dataset,device,snr_grid,mode='awgn',batch=128,max_images=None,mc=1):
    model.eval();rows=[];processed=0
    for item in _as_batch_loader(dataset,batch,device):
        x,_=_get_batch_x_and_names(item,processed)
        if max_images is not None and processed>=max_images:break
        if max_images is not None and processed+len(x)>max_images:x=x[:max_images-processed]
        processed+=len(x);x=x.to(device)
        for s in snr_grid:
            snr=torch.full((len(x),1),float(s),device=device)
            scores=[]
            for rep in range(mc):
                out=fixed_rate_forward(model,x,snr,mode)
                scores.append(psnr_torch(x,out).cpu())
            rows.append({'snr_db':float(s),'sum_psnr':float(torch.stack(scores).mean(0).sum()),'count':len(x)})
    import pandas as pd
    df=pd.DataFrame(rows)
    return (df.groupby('snr_db',as_index=False)[['sum_psnr','count']].sum()
              .assign(mean_psnr_db=lambda x:x.sum_psnr/x['count']))


## 4. Presets and unit sanity checks

The **paper** preset: 48 latent channels, 256 convolutional features, kernel 5, DeepJSCC-V 400 epochs (Adam 1e-4, CIFAR10 batch 128), OraNet 150 epochs (Adam 1e-4), `SNR∈{0,…,27} dB`, `R∈[0.05,0.5]`. The `cached` teacher option is more efficient, but does **not** use new stochastic labels at every OraNet update; set `ORACLE_TRAINING='online'` for the author's online supervised-label approach. Model-selection and test datasets remain separated.

For robustness only, the OraNet final ReLU head is initialized to a positive 25 dB output bias before training; weights and layers are unchanged. Comment out this initialization in the OraNet fit helper for exact upstream random initialization.

The full `paper` setting can exceed **one Colab runtime**. Every training epoch is checkpointed to Google Drive. No GPU performance or total runtime is assumed from the name “G4”; the actual CUDA card is displayed.

In [ ]:
PRESETS={
 'smoke': dict(width=16,kernel=3,train_n=64,val_n=16,c100_n=16,kodak_n=2,
               jscc_epochs=1,oracle_epochs=2,train_batch=8,oracle_batch=16,teacher_batch=8,
               teacher_draws=2,val_draws=2,teacher_mc=1,cifar_mc=1,kodak_mc=1,
               snr_cifar=[6,12],snr_kodak=[6,12],rates_cifar=[1/16,1/8,1/3],
               highres_epochs=1,highres_n=32),
 'research': dict(width=128,kernel=5,train_n=None,val_n=2000,c100_n=2000,kodak_n=24,
                  jscc_epochs=60,oracle_epochs=50,train_batch=64,oracle_batch=512,teacher_batch=64,
                  teacher_draws=8,val_draws=4,teacher_mc=2,cifar_mc=2,kodak_mc=4,
                  snr_cifar=[-3,0,3,6,9,12,15,18,21,24],snr_kodak=[0,3,6,9,12,15,18,21,24],
                  rates_cifar=[1/16,1/14,1/12,1/10,1/8,1/6,1/4,1/3],
                  highres_epochs=15,highres_n=None),
 'paper': dict(width=256,kernel=5,train_n=None,val_n=10000,c100_n=10000,kodak_n=24,
               jscc_epochs=400,oracle_epochs=150,train_batch=128,oracle_batch=512,teacher_batch=64,
               teacher_draws=8,val_draws=4,teacher_mc=2,cifar_mc=1,kodak_mc=100,
               snr_cifar=[-3,0,3,6,9,12,15,18,21,24],snr_kodak=[0,3,6,9,12,15,18,21,24],
               rates_cifar=[1/16,1/14,1/12,1/10,1/8,1/6,1/4,1/3],
               highres_epochs=50,highres_n=None),
}
CFG=PRESETS[PROFILE]
seed_everything(SEED)
DEVICE=device_info()
if PROFILE=='paper' and DEVICE.type!='cuda':
    raise RuntimeError('Select a GPU in Colab: Runtime > Change runtime type > GPU.')
patch_author_attention()
KODAK_RATE_GRID=[1/n for n in range(20,2,-1)] if PROFILE!='smoke' else [1/16,1/12,1/8,1/6,1/4,1/3]
CIFAR_THRESHOLDS=[24,26,28,30,32,34] if PROFILE!='smoke' else [10,20]
KODAK_THRESHOLDS=[28,30,32,34,36,38] if PROFILE!='smoke' else [10,20]
print(json.dumps({k:v for k,v in CFG.items() if k not in ('snr_cifar','snr_kodak','rates_cifar')},indent=2))
# CR conversion checks: paper R=1/16, 1/8, 1/3 -> 6,12,32 channel prefixes.
assert channel_count(torch.tensor([[1/16],[1/8],[1/3]])).reshape(-1).tolist()==[6,12,32]
quick_model=create_jscc(CFG['width'],CFG['kernel'],32,32,device=DEVICE)
quick_x=torch.rand(1,3,32,32,device=DEVICE)
with torch.no_grad():
    quick_out,quick_z,quick_n=jscc_forward(quick_model,quick_x,torch.tensor([[12.]],device=DEVICE),1/8,CHANNEL,return_latent=True)
assert quick_out.shape==quick_x.shape and quick_z.shape==(1,48,8,8) and int(quick_n)==12
assert torch.isfinite(quick_out).all()
print("PASS: author's model, batch=1 attention, 48-channel mask, physical channel, decoder.")
del quick_model,quick_x,quick_out,quick_z,quick_n
if DEVICE.type=='cuda':torch.cuda.empty_cache()
seed_everything(SEED)

## 5. CIFAR10/CIFAR100 data loading

The following cell reads the CIFAR10/CIFAR100 torchvision folders through the MyDrive shortcut; if missing, it **extracts the already uploaded `.tar.gz` archives** from Drive. It does **not** re-extract on every Colab run. No ImageNet or Kodak data is needed for this experiment.


In [ ]:
from pathlib import Path
import tarfile

# Inputs: Shared with me / datasets, reachable via MyDrive/datasets shortcut.
# A shared folder may be read-only: NEVER extract into it.
# If a valid CIFAR directory is present, torchvision loads it directly from the share.
# Only if the extracted directory is missing, expand the original archive to LOCAL scratch.
CIFAR_UNPACK_LOCAL = Path('/content/padc_cifar_shared_unpacked')

def _cifar_files_ok(base, target_folder, required):
    folder = Path(base) / target_folder
    return all((folder / name).is_file() and (folder / name).stat().st_size > 0
               for name in required)

def use_shared_or_local_cifar(root, archive_name, target_folder, required):
    root = Path(root).expanduser()
    if _cifar_files_ok(root, target_folder, required):
        print('Using CIFAR in shared folder (read-only):', root / target_folder)
        return str(root)

    archive = root / archive_name
    if not AUTO_UNPACK_CIFAR_ARCHIVES:
        print(f'CIFAR folder not found in shared source; auto-unpack disabled for {archive_name}')
        return str(root)
    if not archive.is_file():
        print(f'No CIFAR archive at {archive}; .npy / torchvision download overrides may still apply.')
        return str(root)
    if _cifar_files_ok(CIFAR_UNPACK_LOCAL, target_folder, required):
        print('Reusing local CIFAR scratch:', CIFAR_UNPACK_LOCAL / target_folder)
        return str(CIFAR_UNPACK_LOCAL)

    CIFAR_UNPACK_LOCAL.mkdir(parents=True, exist_ok=True)
    print('Read shared archive, unpack to local /content (NOT shared folder):', archive)
    with tarfile.open(archive, mode='r:gz') as tf:
        base = CIFAR_UNPACK_LOCAL.resolve()
        for item in tf.getmembers():
            dest = (CIFAR_UNPACK_LOCAL / item.name).resolve()
            if not dest.is_relative_to(base) or item.issym() or item.islnk():
                raise ValueError('Unsafe archive member: ' + item.name)
        tf.extractall(path=CIFAR_UNPACK_LOCAL, filter='data')
    if not _cifar_files_ok(CIFAR_UNPACK_LOCAL, target_folder, required):
        raise RuntimeError(f'Incomplete CIFAR archive extracted locally: {archive}')
    return str(CIFAR_UNPACK_LOCAL)

if not (CIFAR10_TRAIN_NPY and CIFAR10_TEST_NPY):
    CIFAR10_ROOT = use_shared_or_local_cifar(
        CIFAR10_ROOT,'cifar-10-python.tar.gz','cifar-10-batches-py',
        [f'data_batch_{i}' for i in range(1,6)] + ['test_batch','batches.meta'])
if not CIFAR100_TEST_NPY:
    CIFAR100_ROOT = use_shared_or_local_cifar(
        CIFAR100_ROOT,'cifar-100-python.tar.gz','cifar-100-python',
        ['train','test','meta'])
print('Resolved CIFAR10_ROOT:', CIFAR10_ROOT)
print('Resolved CIFAR100_ROOT:', CIFAR100_ROOT)


In [ ]:
train_raw=cifar_dataset('cifar10','train',CIFAR10_ROOT,CIFAR10_TRAIN_NPY or None,DOWNLOAD_CIFAR_IF_MISSING)
val_raw=cifar_dataset('cifar10','test',CIFAR10_ROOT,CIFAR10_TEST_NPY or None,DOWNLOAD_CIFAR_IF_MISSING)
c100_raw=cifar_dataset('cifar100','test',CIFAR100_ROOT,CIFAR100_TEST_NPY or None,DOWNLOAD_CIFAR_IF_MISSING)
train_set=subset(train_raw,CFG['train_n'],SEED)
val_set=subset(val_raw,CFG['val_n'],SEED+1)
c100_set=subset(c100_raw,CFG['c100_n'],SEED+2)
train_loader=dataloader(train_set,CFG['train_batch'],shuffle=True,workers=2,seed=SEED)
val_loader=dataloader(val_set,CFG['train_batch'],shuffle=False,workers=2,seed=SEED)
# Separate batch sizes for expensive teacher-label generation.
train_teacher_loader=dataloader(train_set,CFG['teacher_batch'],shuffle=False,workers=2,seed=SEED)
val_teacher_loader=dataloader(val_set,CFG['teacher_batch'],shuffle=False,workers=2,seed=SEED)
print('CIFAR10 train/validation:',len(train_set),len(val_set),'CIFAR100 external test:',len(c100_set))
print('Sample shape and range:',train_set[0].shape, (train_set[0].min().item(),train_set[0].max().item()))

## 6. DeepJSCC-V — train or resume the original encoder+decoder

Loss: per-pixel MSE; random per-image SNR and physical CR. After each epoch, save `last` for **resume** and `best` for model selection on CIFAR10 validation. The checkpoint includes the optimizer state and channel identity. Models for AWGN and fading are **not interchangeable**.

In [ ]:
JSCC_DIR=EXPERIMENT_DIR/'checkpoints'
JSCC_BEST=JSCC_DIR/f'jscc_v_{CHANNEL}_cifar10_best.pt'
jscc_cifar=create_jscc(CFG['width'],CFG['kernel'],32,32,device=DEVICE)
if RUN_JSCC_TRAIN:
    info_jscc=fit_jscc(jscc_cifar,train_loader,val_loader,JSCC_DIR,DEVICE,mode=CHANNEL,
                       epochs=CFG['jscc_epochs'],lr=1e-4,resume=True,run_tag='cifar10',seed=SEED)
else:
    info_jscc={'best':str(JSCC_BEST)}
ck_jscc=load_jscc(jscc_cifar,info_jscc['best'],DEVICE,mode=CHANNEL)
JSCC_SHA_CIFAR=hash_file(info_jscc['best'])
jscc_cifar.eval()
print('JSCC BEST:',info_jscc['best'],'SHA256:',JSCC_SHA_CIFAR)

## 7. Freeze DeepJSCC-V and generate OraNet supervision

Per CIFAR10 image sample: draw SNR and physical CR, call the frozen original encoder, keep the **full unmasked** latent, transmit the selected prefix through the channel, decode, compute **uint8 per-image PSNR**. Store a feature-label cache with the SHA256 of **this exact** DeepJSCC-V checkpoint. `cached` greatly reduces repeated GPU inference; the optional `online` setting resamples in each epoch (closer to the author's original training script).

In [ ]:
TEACH_DIR=EXPERIMENT_DIR/'oracle_teacher';TEACH_DIR.mkdir(parents=True,exist_ok=True)
TR_CACHE=TEACH_DIR/f'cifar10_train_{CHANNEL}_{JSCC_SHA_CIFAR[:12]}_d{CFG["teacher_draws"]}_mc{CFG["teacher_mc"]}.npz'
VA_CACHE=TEACH_DIR/f'cifar10_val_{CHANNEL}_{JSCC_SHA_CIFAR[:12]}_d{CFG["val_draws"]}_mc{CFG["teacher_mc"]}.npz'
if ORACLE_TRAINING=='cached':
    build_teacher_table(jscc_cifar,train_teacher_loader,DEVICE,CHANNEL,TR_CACHE,
                        CFG['teacher_draws'],CFG['teacher_mc'],seed=SEED+100,
                        jscc_sha=JSCC_SHA_CIFAR)
build_teacher_table(jscc_cifar,val_teacher_loader,DEVICE,CHANNEL,VA_CACHE,
                    CFG['val_draws'],CFG['teacher_mc'],seed=SEED+200,
                    jscc_sha=JSCC_SHA_CIFAR)
valX,valY,teacher_val_meta=load_teacher(VA_CACHE)
print('Validation teacher dataset:',valX.shape, valY.shape)

## 8. Train OraNet (original residual fully connected blocks)

The input has exactly 98 values: `[CWM(48), CWSTD(48), SNR(dB), active channels]`; target is **PSNR in dB**, loss is MSE. Do not use `Kodak24` or `CIFAR100` as OraNet training data. If changing channel conditions or the DeepJSCC-V checkpoint, the OraNet cache and weights will be isolated by SHA.

In [ ]:
ORACLE_BEST=JSCC_DIR/f'oracle_{CHANNEL}_cifar10_best.pt'
oracle_cifar=create_oracle(DEVICE)
if RUN_ORACLE_TRAIN:
    if ORACLE_TRAINING=='cached':
        info_oracle=fit_oracle_cached(oracle_cifar,TR_CACHE,VA_CACHE,JSCC_DIR,DEVICE,CHANNEL,
                                    epochs=CFG['oracle_epochs'],lr=1e-4,batch=CFG['oracle_batch'],
                                    resume=True,run_tag='cifar10')
    else:
        info_oracle=fit_oracle_online(oracle_cifar,jscc_cifar,train_teacher_loader,VA_CACHE,
                                    JSCC_DIR,DEVICE,CHANNEL,teacher_sha=JSCC_SHA_CIFAR,
                                    epochs=CFG['oracle_epochs'],lr=1e-4,
                                    resume=True,run_tag='cifar10')
else:
    info_oracle={'best':str(ORACLE_BEST)}
load_oracle(oracle_cifar,info_oracle['best'],DEVICE,CHANNEL,JSCC_SHA_CIFAR)
metrics_val=validate_oracle(oracle_cifar,valX,valY,DEVICE)
MARGIN_CIFAR,calib_metrics=empirical_q95_margin(valX,valY,oracle_cifar,DEVICE)
print('OraNet held-out CIFAR10 validation:',{k:round(v,4) for k,v in metrics_val.items() if isinstance(v,float)})
print('Optional one-sided q95 overprediction offset (dB):',round(MARGIN_CIFAR,3))
print('Important: this is an EMPIRICAL offset on CIFAR10, not a mathematical guarantee on CIFAR100/Kodak.')

## 9. ImageNet 2012 validation Parquet → 128×128 transfer → Kodak24 model

The paper uses **45,000 training + 5,000 validation** samples randomly drawn from the **50,000 ImageNet validation** images (labels unused). All images are center-cropped to 128×128. We process the **14 validation Parquet shards** directly, with no need to download/extract `train-*.parquet` or `test-*.parquet`. A persistent lossless uint8 NPY cache is created only once.

**Compute:** The official 50-epoch DeepJSCC-V phase at batch size 32 and a separate 150-epoch online OraNet phase are resource intensive. Leave `IMAGENET_ORACLE_TRAINING='online'` for the more faithful stochastic training. Set to `'cached'` only for a substantially faster, explicitly non-identical fixed-teacher approximation. If CUDA is exhausted, reduce `IMAGENET_TRAIN_BATCH` or `IMAGENET_TEACHER_BATCH`, and record the change.

**Isolation:** CIFAR weights remain unchanged. ImageNet DeepJSCC-V and ImageNet OraNet checkpoints use `imagenet128` run tags. Kodak24 is never used for either training stage.

**Shared folder safety:** Source `validation-*.parquet` is read-only. Persistent RGB128 NPY cache is written to your own `MyDrive/<RUN_FOLDER>/dataset_cache/imagenet128`, not inside shared `datasets/imagenet-1k`.


### 9.1. Parquet reader and permanent center-crop cache

In [ ]:
"""PADC ImageNet 50k validation-Parquet reader, one-time RGB128 lossless cache.

Preserves paper setup: deterministic 45,000/5,000 split, 128x128 center crop.
Never requires ImageNet train/test shards or labels. No Kodak data is used.
"""
import os
import io
import json
import shutil
import hashlib
from pathlib import Path
import numpy as np
import torch
from PIL import Image
from torch.utils.data import Dataset

IMAGENET_EXPECTED_VALIDATION_SHARDS = 14
IMAGENET_EXPECTED_VALIDATION_IMAGES = 50000
IMAGENET_PATCH_SIZE = 128


def imagenet_val_parquet_paths(directory):
    directory = Path(directory)
    paths = sorted(directory.glob('validation-*-of-00014.parquet'))
    expected = [f'validation-{i:05d}-of-00014.parquet' for i in range(14)]
    if [p.name for p in paths] != expected:
        missing = sorted(set(expected) - {p.name for p in paths})
        raise FileNotFoundError(
            f'Expected 14 ImageNet validation Parquet shards in {directory}. '
            f'Found {len(paths)}. Missing: {missing[:14]}. '
            'The 294 train shards and 28 test shards are not required.'
        )
    if not all(p.is_file() and p.stat().st_size > 0 for p in paths):
        raise IOError('One or more validation Parquet shards are empty or missing.')
    return paths


def _parquet_source_manifest(paths):
    return [{'name': p.name, 'bytes': int(p.stat().st_size)} for p in paths]


def prepare_imagenet_val128_npy(parquet_dir, cache_dir, *,
                                expected_images=IMAGENET_EXPECTED_VALIDATION_IMAGES,
                                crop=IMAGENET_PATCH_SIZE, force_rebuild=False,
                                use_local_copy=True):
    """Make a persistent, lossless RGB center-crop NPY (~2.46 GB) from val Parquet.

    Source order: sorted shard filenames, rows within each Parquet shard.
    Uses row-group batches, not a full image data load into RAM.
    Atomic `.incomplete` -> committed .npy and metadata avoids partial cache use.
    Local /content NPY is preferred for multi-epoch training speed, persistent
    copy is kept on Drive for Colab disconnects. No JPEG recompression.
    """
    import pyarrow.parquet as pq
    from torchvision.transforms import CenterCrop

    paths = imagenet_val_parquet_paths(parquet_dir)
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    persistent_file = cache_dir / f'imagenet_validation_{crop}x{crop}_rgb_uint8.npy'
    meta_path = cache_dir / f'imagenet_validation_{crop}x{crop}_rgb_uint8.json'
    local_file = Path('/content') / persistent_file.name
    if not Path('/content').is_dir():
        local_file = cache_dir / '_local_staging' / persistent_file.name
        local_file.parent.mkdir(parents=True, exist_ok=True)
    source_manifest = _parquet_source_manifest(paths)
    requested_meta = {
        'version': 1,
        'source': 'ILSVRC/imagenet-1k validation Parquet shards',
        'source_files': source_manifest,
        'n_images': int(expected_images),
        'crop_size': int(crop),
        'dtype': 'uint8',
        'shape': [int(expected_images), int(crop), int(crop), 3],
        'pixel_pipeline': 'PIL RGB conversion then torchvision CenterCrop; no resizing or re-encoding',
    }
    bytes_required = expected_images * crop * crop * 3
    cache_signature = hashlib.sha256(
        json.dumps(requested_meta, sort_keys=True).encode('utf8')
    ).hexdigest()

    if persistent_file.exists() and meta_path.exists() and not force_rebuild:
        previous_meta = json.loads(meta_path.read_text(encoding='utf8'))
        if previous_meta != requested_meta:
            raise RuntimeError('Existing ImageNet cache metadata does not match source or settings. '
                               'Choose another cache directory or set force_rebuild=True.')
        arr = np.load(persistent_file, mmap_mode='r', allow_pickle=False)
        if tuple(arr.shape) != tuple(requested_meta['shape']) or arr.dtype != np.uint8:
            raise RuntimeError('ImageNet cache is incomplete/corrupt, rebuild it.')
        del arr
        print('Reusing ImageNet 128px persistent NPY cache:', persistent_file)
    else:
        if persistent_file.exists() and not force_rebuild:
            raise RuntimeError(f'Found ImageNet NPY without matching metadata: {persistent_file}. '
                               'Set force_rebuild=True to replace incomplete cache.')
        arrow_rows = sum(pq.ParquetFile(str(p)).metadata.num_rows for p in paths)
        if arrow_rows != expected_images:
            raise ValueError(f'Expected {expected_images:,} ImageNet validation rows, '
                             f'found {arrow_rows:,}. Check Parquet completeness.')
        if local_file.exists():
            local_file.unlink()  # only owned local preparation scratch
        free = shutil.disk_usage(local_file.parent).free
        if free < bytes_required + 512 * 1024**2:
            raise OSError('Insufficient local disk for lossless ImageNet NPY (~2.5 GB). '
                          'Free /content disk, or use a Colab runtime with enough space.')
        print(f'Preparing ImageNet val {arrow_rows:,} images (RGB {crop}x{crop}); '
              'no ImageNet train/test shards used.')
        array = np.lib.format.open_memmap(str(local_file), mode='w+',
                                          dtype=np.uint8,
                                          shape=(expected_images, crop, crop, 3))
        center_crop = CenterCrop((crop, crop))
        written = 0
        try:
            for path in paths:
                pq_file = pq.ParquetFile(str(path))
                if 'image' not in pq_file.schema_arrow.names:
                    raise ValueError(f'Missing `image` column: {path}')
                shard_rows = 0
                for batch in pq_file.iter_batches(columns=['image'], batch_size=128):
                    for val in batch.column(0).to_pylist():
                        if isinstance(val, dict):
                            image_bytes = val.get('bytes')
                            image_path = val.get('path')
                        elif isinstance(val, (bytes, bytearray)):
                            image_bytes, image_path = val, None
                        else:
                            raise TypeError(f'Unsupported `image` entry in {path.name}: {type(val)}')
                        if image_bytes is not None:
                            ctx = Image.open(io.BytesIO(image_bytes))
                        elif image_path and Path(image_path).is_file():
                            ctx = Image.open(image_path)
                        else:
                            raise ValueError(f'No embedded image bytes at {path.name} row {shard_rows}')
                        with ctx as im:
                            rgb = center_crop(im.convert('RGB'))
                            out = np.asarray(rgb, dtype=np.uint8)
                        if out.shape != (crop, crop, 3):
                            raise ValueError(f'Bad ImageNet center crop {path.name}:{shard_rows} {out.shape}')
                        if written >= expected_images:
                            raise ValueError('Too many ImageNet validation images')
                        array[written] = out
                        written += 1
                        shard_rows += 1
                print('Prepared', path.name, f'({shard_rows:,} images); total={written:,}')
            if written != expected_images:
                raise ValueError(f'Incomplete ImageNet cache: {written} images vs {expected_images}')
            array.flush()
        except Exception:
            del array
            if local_file.exists():local_file.unlink()
            raise
        del array
        # Commit only after complete conversion; never mark a partial cache ready.
        persistent_tmp = persistent_file.with_name(persistent_file.name + '.incomplete')
        shutil.copyfile(local_file, persistent_tmp)
        if persistent_tmp.stat().st_size != local_file.stat().st_size:
            raise IOError('ImageNet cache copy truncated')
        os.replace(persistent_tmp, persistent_file)
        meta_tmp = meta_path.with_name(meta_path.name + '.incomplete')
        meta_tmp.write_text(json.dumps(requested_meta, indent=2), encoding='utf8')
        os.replace(meta_tmp, meta_path)
        print('Saved lossless crop cache:', persistent_file)
    if use_local_copy:
        local_sig = local_file.with_name(local_file.name + '.source_sha256')
        local_matches_source = (local_sig.exists() and
                                local_sig.read_text(encoding='utf8') == cache_signature)
        if (not local_file.exists() or
            local_file.stat().st_size != persistent_file.stat().st_size or
            not local_matches_source):
            print('Copying ImageNet RGB128 cache to local SSD for faster training ...')
            tmp = local_file.with_name(local_file.name + '.copying')
            shutil.copyfile(persistent_file, tmp)
            os.replace(tmp, local_file)
            local_sig.write_text(cache_signature, encoding='utf8')
        file_for_training = local_file
    else:
        file_for_training = persistent_file
    data = np.load(file_for_training, mmap_mode='r', allow_pickle=False)
    assert data.shape == (expected_images, crop, crop, 3), data.shape
    sample_0 = data[0]
    print('Prepared dataset:', data.shape, data.dtype,
          'sample pixel range:', int(sample_0.min()), int(sample_0.max()))
    del data
    return file_for_training, requested_meta


class ImageNet128Memmap(Dataset):
    """Lazy mmap numpy crops; exact 45k/5k deterministic split done outside."""
    def __init__(self, npy_path, indices):
        self.npy_path = str(npy_path)
        self.indices = np.asarray(indices, dtype=np.int64)
        self._npy = None
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        if self._npy is None:
            self._npy = np.load(self.npy_path, mmap_mode='r', allow_pickle=False)
        x = np.array(self._npy[int(self.indices[int(i)])], copy=True)
        return torch.from_numpy(x).permute(2, 0, 1).float().div_(255.0)


def imagenet_val128_dataloaders(npy_path, *, seed=2026, train_batch=32,
                                 teacher_batch=16, workers=2, limit=None):
    image_count = len(np.load(npy_path, mmap_mode='r', allow_pickle=False))
    if image_count != 50000:
        raise ValueError(f'Full paper ImageNet validation split requires exactly 50000 images, got {image_count}')
    rng = np.random.default_rng(int(seed))
    perm = rng.permutation(image_count)
    if limit is None:
        tr_idx, va_idx = perm[:45000], perm[45000:]
    else:
        # Explicit debug-only subset. Not comparable with the paper.
        n = min(int(limit), image_count)
        if n < 4: raise ValueError('Need at least 4 images for smoke mode')
        nval = max(1, int(n * 0.1))
        tr_idx, va_idx = perm[:n-nval], perm[n-nval:n]
    if set(tr_idx.tolist()).intersection(va_idx.tolist()):
        raise AssertionError('ImageNet train/validation leakage')
    from torch.utils.data import DataLoader
    def loader(indices, batch, shuffle, salt):
        generator = torch.Generator().manual_seed(int(seed)+int(salt))
        return DataLoader(ImageNet128Memmap(npy_path, indices),
                          batch_size=int(batch), shuffle=bool(shuffle),
                          num_workers=int(workers), pin_memory=torch.cuda.is_available(),
                          persistent_workers=workers > 0, generator=generator)
    tr=loader(tr_idx,train_batch,True,1)
    va=loader(va_idx,train_batch,False,2)
    tr_teacher=loader(tr_idx,teacher_batch,False,3)
    va_teacher=loader(va_idx,teacher_batch,False,4)
    print('ImageNet original validation partition:',len(tr_idx),'train',len(va_idx),'validation')
    print('Patches are lossless RGB 128x128 center crops; no Kodak labels or images.')
    return tr,va,tr_teacher,va_teacher,{
        'seed': int(seed), 'train_n': int(len(tr_idx)), 'val_n': int(len(va_idx)),
        'train_indices_sha256':hashlib.sha256(tr_idx.tobytes()).hexdigest(),
        'val_indices_sha256':hashlib.sha256(va_idx.tobytes()).hexdigest(),
    }


### 9.2. Prepare ImageNet validation patches and fixed train/validation split

In [ ]:
# One-time lossless ImageNet validation -> RGB128 NPY preparation.
# This uses only the 14 validation-*.parquet files; no full ImageNet train/test needed.
if RUN_IMAGENET_FINETUNE:
    import importlib.util
    if importlib.util.find_spec('pyarrow') is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyarrow>=14'], check=True)
    IMNET_NPY, IMNET_CACHE_META = prepare_imagenet_val128_npy(
        IMAGENET_DIR, IMAGENET_CACHE_DIR, expected_images=50000, crop=128,
        force_rebuild=IMAGENET_FORCE_REBUILD,
        use_local_copy=IMAGENET_CACHE_LOCAL,
    )
    high_tr, high_val, high_tt, high_vt, IMNET_SPLIT_META = imagenet_val128_dataloaders(
        IMNET_NPY, seed=SEED+991, train_batch=IMAGENET_TRAIN_BATCH,
        teacher_batch=IMAGENET_TEACHER_BATCH, workers=2,
    )
    # Verify the same image-source and 45k/5k partition on resume.
    imnet_sig = {'source': IMNET_CACHE_META, 'split': IMNET_SPLIT_META,
                 'profile': PROFILE, 'channel': CHANNEL}
    sigfile = EXPERIMENT_DIR / 'imagenet128_data_signature.json'
    if sigfile.exists():
        prior_sig = json.loads(sigfile.read_text(encoding='utf8'))
        if prior_sig != imnet_sig:
            raise RuntimeError('ImageNet dataset/split changed while reusing the same experiment folder. '
                               'Use a new RUN_FOLDER or restore the original SEED/data.')
    else:
        sigfile.write_text(json.dumps(imnet_sig,indent=2),encoding='utf8')
    print('ImageNet training sample:', tuple(next(iter(high_tr)).shape))


In [ ]:
# Full paper-style ImageNet stage. Never contaminate CIFAR or Kodak checkpoints.
jscc_kodak = jscc_cifar
oracle_kodak = oracle_cifar
JSCC_SHA_KODAK = JSCC_SHA_CIFAR
KODAK_SOURCE = 'CIFAR10-only (ImageNet stage disabled)'
ORACLE_SHA_KODAK = hash_file(info_oracle['best'])
if RUN_IMAGENET_FINETUNE:
    if len(high_tr.dataset) != 45000 or len(high_val.dataset) != 5000:
        raise RuntimeError('Paper setting requires EXACT 45k ImageNet train + 5k validation images')
    jscc_kodak = create_jscc(CFG['width'], CFG['kernel'], 128, 128, device=DEVICE)
    jscc_kodak.load_state_dict(jscc_cifar.state_dict(), strict=True)
    hr_out = fit_jscc(
        jscc_kodak, high_tr, high_val, JSCC_DIR, DEVICE, mode=CHANNEL,
        epochs=CFG['highres_epochs'], lr=1e-4, resume=True,
        run_tag='imagenet128', seed=SEED,
    )
    load_jscc(jscc_kodak, hr_out['best'], DEVICE, CHANNEL)
    JSCC_SHA_KODAK = hash_file(hr_out['best'])
    jscc_kodak.eval()
    print('ImageNet 128x128 DeepJSCC-V best:',hr_out['best'])

    tr_hr = TEACH_DIR / f'hrtrain_{CHANNEL}_{JSCC_SHA_KODAK[:12]}_d{CFG["teacher_draws"]}_mc{CFG["teacher_mc"]}.npz'
    va_hr = TEACH_DIR / f'hrval_{CHANNEL}_{JSCC_SHA_KODAK[:12]}_d{CFG["val_draws"]}_mc{CFG["teacher_mc"]}.npz'
    if IMAGENET_ORACLE_TRAINING == 'cached':
        build_teacher_table(jscc_kodak, high_tt, DEVICE, CHANNEL, tr_hr,
                            CFG['teacher_draws'], CFG['teacher_mc'],
                            seed=SEED+401, jscc_sha=JSCC_SHA_KODAK)
    build_teacher_table(jscc_kodak, high_vt, DEVICE, CHANNEL, va_hr,
                        CFG['val_draws'], CFG['teacher_mc'],
                        seed=SEED+402, jscc_sha=JSCC_SHA_KODAK)

    oracle_kodak = create_oracle(DEVICE)
    if IMAGENET_ORACLE_TRAINING == 'cached':
        hr_oracle = fit_oracle_cached(
            oracle_kodak, tr_hr, va_hr, JSCC_DIR, DEVICE, CHANNEL,
            epochs=IMAGENET_ORACLE_EPOCHS, batch=CFG['oracle_batch'],
            lr=1e-4, resume=True, run_tag='imagenet128')
    else:
        hr_oracle = fit_oracle_online(
            oracle_kodak, jscc_kodak, high_tt, va_hr, JSCC_DIR, DEVICE, CHANNEL,
            JSCC_SHA_KODAK, epochs=IMAGENET_ORACLE_EPOCHS,
            lr=1e-4, resume=True, run_tag='imagenet128')
    load_oracle(oracle_kodak, hr_oracle['best'], DEVICE, CHANNEL, JSCC_SHA_KODAK)
    ORACLE_SHA_KODAK = hash_file(hr_oracle['best'])
    val_hr_X, val_hr_Y, _ = load_teacher(va_hr)
    hr_val_metrics = validate_oracle(oracle_kodak, val_hr_X, val_hr_Y, DEVICE)
    print('ImageNet OraNet val:', {k:round(v,4) for k,v in hr_val_metrics.items() if isinstance(v,float)})
    KODAK_SOURCE = ('ImageNet validation (45k/5k), RGB128 center crop, '
                    f'JSCC 50 epochs, OraNet {IMAGENET_ORACLE_EPOCHS} epochs '
                    f'({IMAGENET_ORACLE_TRAINING})')
print('Kodak evaluation model:', KODAK_SOURCE)


## 10. CIFAR10 data-level calibration and CIFAR100 **held-out** test

Reproduce (conceptually) Fig. 4/5 and Fig. 7. Data-level: from **CIFAR10 validation only**, select ONE CR meeting the **mean PSNR threshold** at each SNR. Instance-level: from OraNet image content predictions, select minimum predicted-feasible CR. The **genie** benchmark is chosen **after observing all test-grid PSNRs** and is *not* a feasible transmitter. `test_satisfied` directly measures quality-constraint violations.

The grid CSV is saved batchwise with a configuration sidecar, and evaluation **resumes** after a Colab disconnect. A trained model's SHA is checked before appending.

In [ ]:
if RUN_CIFAR_EVALUATION:
    cifar_val_grid=evaluate_grid(jscc_cifar,oracle_cifar,val_set,'CIFAR10_validation',
        CFG['snr_cifar'],CFG['rates_cifar'],DEVICE,CHANNEL,mc_repeats=CFG['cifar_mc'],
        batch=(8 if PROFILE=='smoke' else 64),
        output_csv=RESULTS_DIR/'cifar10_val_grid.csv',seed=SEED+501,model_sha=JSCC_SHA_CIFAR)
    cifar100_grid=evaluate_grid(jscc_cifar,oracle_cifar,c100_set,'CIFAR100_test',
        CFG['snr_cifar'],CFG['rates_cifar'],DEVICE,CHANNEL,mc_repeats=CFG['cifar_mc'],
        batch=(8 if PROFILE=='smoke' else 64),
        output_csv=RESULTS_DIR/'cifar100_test_grid.csv',seed=SEED+502,model_sha=JSCC_SHA_CIFAR)
    data_lookup=lookup_data_level(cifar_val_grid,CIFAR_THRESHOLDS,CFG['snr_cifar'])
    data_lookup.to_csv(RESULTS_DIR/'cifar_data_level_choices.csv',index=False)
    cifar_opt=optimize_from_grid(cifar100_grid,CIFAR_THRESHOLDS,data_lookup,conservative_margin=0.)
    cifar_opt_calibrated=optimize_from_grid(cifar100_grid,CIFAR_THRESHOLDS,data_lookup,
                                            conservative_margin=MARGIN_CIFAR)
    cifar_opt.to_csv(RESULTS_DIR/'cifar100_instance_padc.csv',index=False)
    cifar_opt_calibrated.to_csv(RESULTS_DIR/'cifar100_padc_calibrated.csv',index=False)
    summary_cifar=report_metrics(cifar100_grid,cifar_opt)
    display(pd.DataFrame([summary_cifar]))
    display(summarize_optimizer(cifar_opt))
    print('Note: CIFAR100 is external test data; no weights were fitted to CIFAR100.')

In [ ]:
plot_rate_selection(
    cifar_opt_calibrated,
    'CIFAR100_test',
    FIG_DIR,
    tag='_calibrated'
)

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PADC_ONLY_DIR = RESULTS_DIR / 'padc_only_paper_figures'
PADC_ONLY_DIR.mkdir(parents=True, exist_ok=True)

# 기존 CSV 재사용
if 'cifar100_grid' not in globals():
    cifar100_grid = pd.read_csv(
        RESULTS_DIR / 'cifar100_test_grid.csv'
    )

d = cifar100_grid.copy()

rate_map = {
    6: '1/16',
    8: '1/12',
    16: '1/6',
    32: '1/3'
}

# ========================================
# Fig. 4(a): DeepJSCC-V PSNR vs SNR
# ========================================

fig, ax = plt.subplots(figsize=(7, 4.5))

for ch, label in rate_map.items():
    g = (
        d[d.active_channels.eq(ch)]
        .groupby('snr_db', as_index=False)['psnr_db']
        .mean()
        .sort_values('snr_db')
    )

    ax.plot(
        g.snr_db, g.psnr_db,
        '-o', label=f'DeepJSCC-V, R={label}'
    )

ax.set(
    xlabel='Channel SNR (dB)',
    ylabel='Average PSNR (dB)',
    title='Fig. 4(a)-like: CIFAR100 / AWGN'
)
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()

fig.savefig(
    PADC_ONLY_DIR / 'fig4_deepjscc_v_only.png',
    dpi=180
)
plt.show()


# ========================================
# Fig. 5(a): Data-level minimum CR
# ========================================

av = (
    d.groupby(['snr_db', 'effective_r'], as_index=False)
    ['psnr_db'].mean()
    .sort_values(['snr_db', 'effective_r'])
)

fig5_rows = []

for snr, group in av.groupby('snr_db'):
    for target in [24, 26, 28, 30, 32, 34]:

        feasible = group[group.psnr_db >= target]

        fig5_rows.append({
            'snr_db': snr,
            'target_psnr_db': target,
            'min_r': (
                float(feasible.iloc[0].effective_r)
                if len(feasible) else np.nan
            ),
            'feasible': bool(len(feasible))
        })

fig5_df = pd.DataFrame(fig5_rows)

fig5_df.to_csv(
    PADC_ONLY_DIR / 'fig5_deepjscc_v_data_level.csv',
    index=False
)

fig, ax = plt.subplots(figsize=(7, 4.5))

for target, g in fig5_df.groupby('target_psnr_db'):
    ax.plot(
        g.snr_db, g.min_r,
        '-o', label=f'Target {target:g} dB'
    )

ax.set(
    xlabel='Channel SNR (dB)',
    ylabel='Minimum physical CR',
    title='Fig. 5(a)-like: DeepJSCC-V Data-level CR'
)

ax.grid(alpha=0.3)
ax.legend(ncol=2)
fig.tight_layout()

fig.savefig(
    PADC_ONLY_DIR / 'fig5_deepjscc_v_only.png',
    dpi=180
)
plt.show()

print(
    'Infeasible target conditions:',
    int((~fig5_df.feasible).sum())
)


# ========================================
# Fig. 7(a): OraNet prediction MSE
# ========================================

fig, ax = plt.subplots(figsize=(7, 4.5))

for ch in [6, 8, 16]:

    g = d[
        (d.active_channels == ch) &
        (d.snr_db >= 0)
    ].copy()

    g['sq_error'] = (
        g.predicted_psnr_db - g.psnr_db
    ) ** 2

    p = (
        g.groupby('snr_db', as_index=False)['sq_error']
        .mean()
    )

    ax.plot(
        p.snr_db, p.sq_error,
        '-o', label=f'R={rate_map[ch]}'
    )

ax.set(
    xlabel='Channel SNR (dB)',
    ylabel='PSNR prediction MSE (dB²)',
    title='Fig. 7(a): CIFAR100 OraNet / AWGN'
)

ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()

fig.savefig(
    PADC_ONLY_DIR / 'fig7_oranet_mse_paper_rates.png',
    dpi=180
)
plt.show()

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Subset

jscc_cifar.eval()

random_ids = np.random.default_rng(SEED).choice(
    len(c100_set), size=3, replace=False
).tolist()

three = Subset(c100_set, random_ids)

fig6_grid = evaluate_grid(
    jscc_cifar,
    None,
    three,
    'CIFAR100_Fig6_3images',
    [12.0],
    [k / 96 for k in range(5, 49)],
    DEVICE,
    CHANNEL,
    mc_repeats=10,
    batch=3,
    seed=SEED + 806,
    output_csv=RESULTS_DIR / 'fig6_3images_dense_cr.csv',
    model_sha=JSCC_SHA_CIFAR
)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for j, original_idx in enumerate(random_ids):

    x = c100_set[original_idx].unsqueeze(0).to(DEVICE)
    snr = torch.full((1, 1), 12., device=DEVICE)

    with torch.no_grad():
        z = encode_features(jscc_cifar, x, snr)

        mu = z.mean(dim=(2, 3))[0].cpu().numpy()
        std = z.std(
            dim=(2, 3), unbiased=True
        )[0].cpu().numpy()

    label = f'Image {j+1} (idx={original_idx})'

    axes[0].plot(np.arange(1, 49), mu, label=label)
    axes[1].plot(np.arange(1, 49), std, label=label)

    g = fig6_grid[
        fig6_grid.image_id.eq(f'img_{j:05d}')
    ].sort_values('effective_r')

    axes[2].plot(
        g.effective_r, g.psnr_db,
        label=label
    )

axes[0].set_title('(a) Channel-wise Mean')
axes[0].set_xlabel('Channel ID')

axes[1].set_title('(b) Channel-wise STD')
axes[1].set_xlabel('Channel ID')

axes[2].set_title('(c) PSNR vs CR, SNR=12 dB')
axes[2].set_xlabel('Physical CR')
axes[2].set_ylabel('PSNR (dB)')

for ax in axes:
    ax.grid(alpha=0.3)

axes[0].legend(fontsize=7)
fig.tight_layout()

fig.savefig(
    PADC_ONLY_DIR / 'fig6_three_cifar100_examples.png',
    dpi=180
)
plt.show()

In [ ]:
if RUN_CIFAR_EVALUATION:
    plot_psnr_grid(cifar100_grid,'CIFAR100_test',FIG_DIR,title_suffix=f'({CHANNEL})')
    plot_oracle_error(cifar100_grid,'CIFAR100_test',FIG_DIR)
    plot_rate_selection(cifar_opt,'CIFAR100_test',FIG_DIR,tag='_uncalibrated')
    plot_rate_selection(cifar_opt_calibrated,'CIFAR100_test',FIG_DIR,tag='_calibrated')

## 11. Kodak24 out-of-distribution evaluation, OraNet PSNR prediction, per-image CR optimization

Kodak24 has 24 RGB images commonly sized 768×512 or 512×768. Model weights work across shapes; the decoder's **spatial view is adjusted** to the current image, which is required by the author's original network. Do not upsample CIFAR to pretend to have performed the ImageNet phase. With CIFAR10-only weights, Kodak results are explicitly labeled an **out-of-domain transfer test**.

- Fig. 11 style: actual vs predicted PSNR at `R={1/16,1/12,1/8,1/6,1/4}` across SNR.
- Fig. 13 style: image-level exhaustive CR search over nominal `{1/20,1/19,…,1/3}`. Several nominal rates map to the same *48-channel* mask, so deduplicate identical **effective rates** (no bandwidth distinction) before measuring.
- The paper's Kodak metric uses **100 channel realizations**, selected by the `paper` preset. **Warning:** this can be very computationally expensive on full-resolution Kodak; incremental CSV checkpoints permit restart. Start with `smoke`/`research` to check memory and rate.

In [ ]:
# Shared Kodak24 is a read-only dataset input. Never mkdir or download into the share.
if RUN_KODAK_EVALUATION:
    from pathlib import Path
    import shutil
    from urllib.request import urlretrieve
    from PIL import Image
    source_folder = Path(KODAK_PATH)
    if not source_folder.is_dir():
        raise FileNotFoundError('Shared Kodak24 folder missing: '+str(source_folder))
    missing = [i for i in range(1,25) if not (source_folder/f'kodim{i:02d}.png').is_file()]
    if missing == [18]:
        local_kodak = Path('/content/padc_kodak24_completed')
        local_kodak.mkdir(parents=True,exist_ok=True)
        for i in range(1,25):
            if i != 18:
                shutil.copy2(source_folder/f'kodim{i:02d}.png',local_kodak/f'kodim{i:02d}.png')
        target = local_kodak / 'kodim18.png'
        urlretrieve('https://r0k.us/graphics/kodak/kodak/kodim18.png',str(target))
        with Image.open(target) as img: img.verify()
        KODAK_PATH = str(local_kodak)
        print('Kodak missing 18 restored to LOCAL scratch; shared folder was not modified.')
        missing = []
    print('Missing Kodak24 images:', missing)
    if missing:
        raise FileNotFoundError(f'Kodak24 must contain 24 PNG files, missing={missing}')


In [ ]:
if RUN_KODAK_EVALUATION:
    kodak_paths=resolve_kodak(KODAK_PATH,extracted_dir=Path('/content/padc_kodak_unpacked'),require_24=True)
    kodak_full=Kodak24(kodak_paths)
    kodak_set=subset(kodak_full,CFG['kodak_n'],SEED+3)
    print('Kodak24 paths found:',len(kodak_full),'evaluating:',len(kodak_set))
    print('First Kodak:',kodak_full[0][1],tuple(kodak_full[0][0].shape),'Weight source:',KODAK_SOURCE)
    kodak_grid=evaluate_grid(jscc_kodak,oracle_kodak,kodak_set,'Kodak24_test',
        CFG['snr_kodak'],KODAK_RATE_GRID,DEVICE,CHANNEL,mc_repeats=KODAK_MC_REPEATS,
        batch=1,output_csv=RESULTS_DIR/f'kodak24_imagenet128_{CHANNEL}_mc{KODAK_MC_REPEATS}_full_rate_grid.csv',
        seed=SEED+600,model_sha=JSCC_SHA_KODAK,oracle_sha=ORACLE_SHA_KODAK)
    kodak_opt=optimize_from_grid(kodak_grid,KODAK_THRESHOLDS,
                                 data_lookup=None,conservative_margin=0.)
    kodak_opt.to_csv(RESULTS_DIR/f'kodak24_imagenet128_{CHANNEL}_mc{KODAK_MC_REPEATS}_instance_padc.csv',index=False)
    summary_kodak=report_metrics(kodak_grid,kodak_opt)
    display(pd.DataFrame([summary_kodak]))
    display(summarize_optimizer(kodak_opt))
    print('Evaluation note:',KODAK_SOURCE)

In [ ]:
if RUN_KODAK_EVALUATION:
    # Only the 5 paper Fig.11 rates for a legible PSNR/prediction plot.
    fig11_nominal=[1/16,1/12,1/8,1/6,1/4]
    fig11_active={int(channel_count(r).item()) for r in fig11_nominal}
    kodak_fig11=kodak_grid[kodak_grid.active_channels.isin(fig11_active)].copy()
    plot_psnr_grid(kodak_fig11,'Kodak24_test',FIG_DIR,title_suffix=f'({CHANNEL}, Fig.11-like)')
    plot_oracle_error(kodak_fig11,'Kodak24_test',FIG_DIR)
    plot_rate_selection(kodak_opt,'Kodak24_test',FIG_DIR)
    for img in ['kodim13','kodim23']:
        if img in set(kodak_opt.image_id):
            plot_rate_selection(kodak_opt,'Kodak24_test',FIG_DIR,image=img)

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

# ========================================
# Fig. 8: DeepJSCC-V only
# ========================================

fig, ax = plt.subplots(figsize=(7, 4.5))

for ch, rate in [(16, '1/6'), (8, '1/12')]:

    q = (
        kodak_grid[
            kodak_grid.active_channels.eq(ch)
        ]
        .groupby('snr_db', as_index=False)['psnr_db']
        .mean()
        .sort_values('snr_db')
    )

    ax.plot(
        q.snr_db, q.psnr_db,
        '-o', label=f'DeepJSCC-V, R={rate}'
    )

ax.set(
    xlabel='SNR (dB)',
    ylabel='Kodak24 average PSNR (dB)',
    title='Fig. 8-style: DeepJSCC-V only'
)

ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()

fig.savefig(
    PADC_ONLY_DIR / 'fig8_jscc_only.png',
    dpi=180
)
plt.show()


# ========================================
# Fig. 10: Kodak image PSNR vs CR
# ========================================

fig, ax = plt.subplots(figsize=(7, 4.5))

for name in ['kodim08', 'kodim21', 'kodim24']:

    q = kodak_grid[
        (kodak_grid.image_id == name) &
        (np.isclose(kodak_grid.snr_db, 12.))
    ].sort_values('effective_r')

    ax.plot(
        q.effective_r,
        q.psnr_db,
        '-o',
        label=name
    )

ax.set(
    xlabel='Physical CR (K/N)',
    ylabel='PSNR (dB)',
    title='Fig. 10-like: Kodak24, AWGN 12 dB'
)

ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()

fig.savefig(
    PADC_ONLY_DIR / 'fig10_kodak_three_images.png',
    dpi=180
)
plt.show()


# ========================================
# Fig. 13: Predicted vs tested optimal CR
# ========================================

small_kodak = kodak_grid[
    kodak_grid.image_id.isin(['kodim13', 'kodim23'])
]

fig13 = optimize_from_grid(
    small_kodak,
    [26, 28, 30, 34, 36, 38],
    data_lookup=None
)

fig13.to_csv(
    PADC_ONLY_DIR / 'fig13_kodak_predicted_tested_CR.csv',
    index=False
)

for name, targets in [
    ('kodim13', [26, 28, 30]),
    ('kodim23', [30, 34, 36, 38])
]:

    fig, ax = plt.subplots(figsize=(7, 4.5))

    for th in targets:

        q = fig13[
            (fig13.image_id == name) &
            (fig13.target_psnr_db == th)
        ].sort_values('snr_db')

        # Feasible 조건만 표시
        actual = q.genie_r.where(
            q.genie_test_feasible
        )

        pred = q.padc_r.where(
            q.padc_prediction_feasible
        )

        line = ax.plot(
            q.snr_db, actual,
            '-o', label=f'Tested optimal, {th} dB'
        )[0]

        ax.plot(
            q.snr_db, pred,
            '--x',
            color=line.get_color(),
            label=f'OraNet selected, {th} dB'
        )

    ax.set(
        xlabel='Channel SNR (dB)',
        ylabel='Optimal physical CR',
        title=f'Fig. 13-like: {name} / AWGN'
    )

    ax.grid(alpha=0.3)
    ax.legend(fontsize=8, ncol=2)
    fig.tight_layout()

    fig.savefig(
        PADC_ONLY_DIR / f'fig13_{name}.png',
        dpi=180
    )
    plt.show()

## 12. Reconstruction images & feature inspection

This visualization uses exactly ONE channel realization; its PSNR will vary from the Monte Carlo mean reported in the grid. `OraNet` predicts a statistical quality value from unmasked features, not the exact realization of random receiver noise.

In [ ]:
if RUN_KODAK_EVALUATION:
    kodak_names=[kodak_full[i][1] for i in range(len(kodak_full))]
    selected_ids=[13,23] if PROFILE!='smoke' else [1,2]
    for idx in selected_ids:
        if idx<=len(kodak_full):
            ex=reconstruct_example(jscc_kodak,oracle_kodak,kodak_full,idx-1,12.0,1/8,DEVICE,CHANNEL)
            plot_example(ex,FIG_DIR)

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np

@torch.no_grad()
def reconstruct_one_sc_channel(
    image_no,
    channel_no,
    snr_db=12.0
):
    x, _ = kodak_full[image_no - 1]
    x = x.unsqueeze(0).to(DEVICE)

    snr = torch.tensor(
        [[snr_db]], device=DEVICE
    )

    z = encode_features(
        jscc_kodak, x, snr
    )

    # 특정 SC 채널 하나만 활성화
    mask = torch.zeros_like(z)
    mask[:, channel_no - 1] = 1.0

    selected = (z * mask).flatten(1)
    mask_flat = mask.flatten(1)

    n_real = z.shape[-2] * z.shape[-1]

    norm = selected * (
        n_real /
        selected.square()
        .sum(dim=1, keepdim=True)
        .clamp_min(1e-8)
    ).sqrt()

    # AWGN 전송
    with torch_fixed_rng(
        SEED + image_no * 100 + channel_no,
        DEVICE
    ):
        received = (
            norm +
            torch.randn_like(norm) *
            (10 ** (-snr_db / 20)) *
            mask_flat
        ) * mask_flat

    jscc_kodak.decoder.enc_shape = [
        48, z.shape[-2], z.shape[-1]
    ]

    y = jscc_kodak.decoder(
        received, snr
    ).clamp(0, 1)

    psnr = float(
        psnr_torch(
            x, y, quantized=True
        ).item()
    )

    return (
        y[0].permute(1, 2, 0).cpu().numpy(),
        psnr
    )


# ========================================
# Fig. 9: kodim08, 21, 24
# ========================================

fig, axes = plt.subplots(
    3, 3, figsize=(12, 9)
)

for row, image_no in enumerate([8, 21, 24]):

    x, name = kodak_full[image_no - 1]

    axes[row, 0].imshow(
        x.permute(1, 2, 0).numpy()
    )
    axes[row, 0].set_title(
        f'{name}: original'
    )

    for col, channel_no in enumerate(
        [5, 7], start=1
    ):
        img, psnr = reconstruct_one_sc_channel(
            image_no, channel_no
        )

        axes[row, col].imshow(img)
        axes[row, col].set_title(
            f'Channel {channel_no}: {psnr:.2f} dB'
        )

for ax in axes.flat:
    ax.axis('off')

fig.suptitle(
    'Fig. 9-like: Single SC channel, AWGN 12 dB'
)
fig.tight_layout()

fig.savefig(
    PADC_ONLY_DIR / 'fig9_single_channel_ablation.png',
    dpi=160
)
plt.show()

## 13. Optional ADJSCC baseline — fixed-rate original network

Not required for a complete PADC system, but useful to compare Fig.4 fairly: original `models.ADJSCC` (same residual/attention architecture without prefix masking), independently train four fixed-rate models for `R = {1/16, 1/12, 1/6, 1/3}` on **CIFAR10**. Uses identical channels and PSNR conventions, but training four more models is costly and may require additional Colab sessions.

In [ ]:
if RUN_ADJSCC_BASELINES:
    BASELINE_RATES=[1/16,1/12,1/6,1/3]
    baseline_rows=[]
    for br in BASELINE_RATES:
        bmodel=create_adjscc_fixed(br,CFG['width'],CFG['kernel'],DEVICE)
        bpath=fit_adjscc_fixed(bmodel,br,train_loader,val_loader,JSCC_DIR,DEVICE,mode=CHANNEL,
                                epochs=(1 if PROFILE=='smoke' else 200),lr=1e-4,resume=True,
                                max_batches=(2 if PROFILE=='smoke' else None))
        curve=evaluate_adjscc_fixed(bmodel,c100_set,DEVICE,CFG['snr_cifar'],CHANNEL,
                                    batch=(8 if PROFILE=='smoke' else 64),
                                    mc=CFG['cifar_mc'])
        curve['requested_r']=br
        curve['active_channels']=int(channel_count(br).item())
        baseline_rows.append(curve)
        del bmodel
        if DEVICE.type=='cuda':torch.cuda.empty_cache()
    baseline_df=pd.concat(baseline_rows,ignore_index=True)
    baseline_df.to_csv(RESULTS_DIR/'cifar100_ADJSCC_baselines.csv',index=False)
    display(baseline_df.head(12))

## 14. Reproducibility manifest, artifacts and interpretation

Interpreting optimization results:

- `padc_prediction_feasible=False` means *no tested CR is predicted to reach the target*; we record the **maximum** CR as a fallback **not** as successful optimization.
- `padc_test_satisfied=False` is a **measured** threshold violation, even if the predictor said feasible.
- `excess_r_vs_genie` is computed ONLY for instances where PADC meets the target and the genie is feasible; failed reconstructions do NOT count as bandwidth savings.
- `genie_r` uses the test image's own *measured* PSNR across rates **after transmission**, hence is an **oracle lower-bound comparison only**, not an implementable encoder decision.
- `data_level_r` meets a *validation-set average* target, **not a per-image constraint**; it is only an appropriate comparator on the CIFAR100 grid.
- A one-sided calibration margin is **empirical** and cannot guarantee individual Kodak images under domain shift.

The output directory contains best/last weights, teacher-label caches, grid CSVs, optimal-rate CSVs, plots, and manifest JSON. Copy or compress that directory from Drive for a new Colab session. No metrics are filled in before models actually run.

In [ ]:
from datetime import datetime, timezone
manifest={
  'created_at_utc':datetime.now(timezone.utc).isoformat(),
  'paper':'Zhang et al., Predictive and Adaptive Deep Coding for Wireless Image Transmission in Semantic Communication, TWC 2023',
  'paper_doi':'10.1109/TWC.2023.3234408',
  'upstream_git_sha':AUTHOR_SHA,'upstream_git_url':AUTHOR_REPO_URL,
  'python':sys.version,'pytorch':torch.__version__,
  'cuda_gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
  'profile':PROFILE,'channel':CHANNEL,'profile_config':CFG,
  'run_folder':RUN_FOLDER,
  'training_source':'CIFAR10','holdout_cifar':'CIFAR100 test','holdout_kodak':('Kodak24' if RUN_KODAK_EVALUATION else None),
  'kodak_transfer':KODAK_SOURCE,'jscc_cifar_sha256':JSCC_SHA_CIFAR,
  'imagenet_validation_parquet_dir':IMAGENET_DIR if RUN_IMAGENET_FINETUNE else None,
  'imagenet_split_metadata':IMNET_SPLIT_META if RUN_IMAGENET_FINETUNE else None,
  'imagenet_persistent_crop_cache':IMAGENET_CACHE_DIR if RUN_IMAGENET_FINETUNE else None,
  'imagenet_oracle_training_mode':IMAGENET_ORACLE_TRAINING if RUN_IMAGENET_FINETUNE else None,
  'imagenet_oracle_epochs':IMAGENET_ORACLE_EPOCHS if RUN_IMAGENET_FINETUNE else None,
  'imagenet_oracle_sha256':ORACLE_SHA_KODAK if RUN_IMAGENET_FINETUNE else None,
  'kodak_mc_repeats':KODAK_MC_REPEATS if RUN_KODAK_EVALUATION else None,
  'jscc_kodak_sha256':JSCC_SHA_KODAK,
  'oracle_training_mode':ORACLE_TRAINING,
  'cr_semantics':'R=K/(3HW), repo internal cr=2R, effective R=round(96R)/96',
  'psnr':'uint8 per-image MSE/PSNR; repo-compatible truncated uint8 pixels',
  'test_metrics_cifar':(summary_cifar if RUN_CIFAR_EVALUATION else None),
  'test_metrics_kodak':(summary_kodak if RUN_KODAK_EVALUATION else None),
  'note':'No physical guarantee from a regressor; empirical constraint rates reported.',
}
(RESULTS_DIR/'manifest.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf8')
print('Finished. Results:',RESULTS_DIR)
print('Full experiment directory:',EXPERIMENT_DIR)
print('To download a compact copy of results *excluding giant model checkpoints* in Colab:')
print('  shutil.make_archive("/content/PADC_results", "zip", root_dir=str(RESULTS_DIR))')